## NammaMart ER Diagram

![NammaMart ER Diagram](../Data_profile_relationship_diagram.png)

# Stage 1: Initial Data Inspection

This stage only loads the source files and inspects their basic structure. It does not clean, modify, standardise, impute, drop, or classify any values.

## 1. Import libraries

This cell imports pandas for table loading and inspection, plus `Path` for readable file paths. Importing libraries confirms the tools used in this stage; it does not inspect or validate the data.

In [1]:
import pandas as pd
from pathlib import Path

## 2. Define the dataset directory

This cell stores the folder containing the six source CSV files in one variable. Keeping the location central makes the loading code easier to read and change; it does not confirm that every expected file exists.

In [2]:
dataset_dir = Path(".")

## 3. Load all six source files

This cell reads every CSV without type conversion by using `dtype=str` and without treating any text as a missing value. That is useful for preserving raw identifiers and formatting; it does not tell us whether values are complete, valid, unique, or correct.

In [3]:
csv_options = {
    "dtype": str,
    "keep_default_na": False,
}

customers_df = pd.read_csv(dataset_dir / "customers.csv", **csv_options)
orders_df = pd.read_csv(dataset_dir / "orders.csv", **csv_options)
payments_df = pd.read_csv(dataset_dir / "payments.csv", **csv_options)
products_df = pd.read_csv(dataset_dir / "products.csv", **csv_options)
stores_df = pd.read_csv(dataset_dir / "stores.csv", **csv_options)
orders_increment_df = pd.read_csv(
    dataset_dir / "orders_2026-10-01_increment.csv",
    **csv_options,
)

## 4. Confirm that the expected files were loaded

This assertion checks that all six DataFrame variables exist and that each source path is present. It catches a missing file or a misspelled variable name; it does not prove that the rows themselves are trustworthy.

In [4]:
dataframes = {
    "customers_df": customers_df,
    "orders_df": orders_df,
    "payments_df": payments_df,
    "products_df": products_df,
    "stores_df": stores_df,
    "orders_increment_df": orders_increment_df,
}

expected_files = {
    "customers.csv",
    "orders.csv",
    "payments.csv",
    "products.csv",
    "stores.csv",
    "orders_2026-10-01_increment.csv",
}

assert expected_files == {
    path.name for path in dataset_dir.glob("*.csv")
}, "The dataset directory does not contain exactly the six expected CSV files."
assert len(dataframes) == 6, "All six DataFrames must be defined."
assert all(isinstance(dataframe, pd.DataFrame) for dataframe in dataframes.values())
assert all(not dataframe.empty for dataframe in dataframes.values()), (
    "None of the six DataFrames should be empty."
)

print("All six files loaded and all six DataFrames contain at least one row.")

All six files loaded and all six DataFrames contain at least one row.


## 5. Build a dataset inventory

This inventory gives a first structural overview of each loaded table, including row and column counts, exact duplicate full rows, and estimated memory use. It helps us understand dataset size and shape; it does not identify bad values or duplicate keys.

In [5]:
file_names = {
    "customers_df": "customers.csv",
    "orders_df": "orders.csv",
    "payments_df": "payments.csv",
    "products_df": "products.csv",
    "stores_df": "stores.csv",
    "orders_increment_df": "orders_2026-10-01_increment.csv",
}

inventory_rows = []
for dataframe_name, dataframe in dataframes.items():
    inventory_rows.append(
        {
            "file_name": file_names[dataframe_name],
            "dataframe_name": dataframe_name,
            "rows": len(dataframe),
            "columns": len(dataframe.columns),
            "duplicate_full_row_count": int(dataframe.duplicated().sum()),
            "estimated_memory_bytes": int(dataframe.memory_usage(deep=True).sum()),
        }
    )

inventory_df = pd.DataFrame(inventory_rows)
inventory_df

,file_name,dataframe_name,rows,columns,duplicate_full_row_count,estimated_memory_bytes
0,customers.csv,customers_df,405,10,0,237533
1,orders.csv,orders_df,2412,13,12,1713791
2,payments.csv,payments_df,2228,7,4,910980
3,products.csv,products_df,152,10,0,87582
4,stores.csv,stores_df,15,8,0,7038
5,orders_2026-10-01_increment.csv,orders_increment_df,160,13,0,114113


## 6. Inspect `orders.csv`

These views show the order table's shape, source column order, inferred pandas dtypes, and representative rows. They help us see the raw structure and sample contents; they do not prove that the schema, values, keys, or relationships are correct.

In [6]:
print("Shape:", orders_df.shape)
print("\nColumn names in source order:")
print(orders_df.columns.tolist())

print("\nPandas-inferred data types after conservative loading:")
print(orders_df.dtypes)

print("\nFirst 5 rows:")
display(orders_df.head(5))

print("\nLast 5 rows:")
display(orders_df.tail(5))

print("\n5 random rows with fixed random_state=42:")
display(orders_df.sample(n=5, random_state=42))

Shape: (2412, 13)

Column names in source order:
['order_id', 'order_ts', 'customer_id', 'store_id', 'product_id', 'quantity', 'unit_price', 'discount_pct', 'order_amount', 'payment_mode', 'order_status', 'channel', 'delivery_minutes']

Pandas-inferred data types after conservative loading:
order_id            str
order_ts            str
customer_id         str
store_id            str
product_id          str
quantity            str
unit_price          str
discount_pct        str
order_amount        str
payment_mode        str
order_status        str
channel             str
delivery_minutes    str
dtype: object

First 5 rows:


,order_id,order_ts,customer_id,store_id,product_id,quantity,unit_price,discount_pct,order_amount,payment_mode,order_status,channel,delivery_minutes
0,ORD001913,2026-09-24 22:56:41,C00005,S10,P0023,2,222.0,5,421.8,UPI,DELIVERED,App,35
1,ORD001791,2026-09-23 17:54:42,C00369,S05,P0136,1,360.0,0,360.0,UPI,DELIVERED,App,31
2,ORD002203,2026-09-28 15:53:19,C00010,S08,P0135,1,248.0,0,248.0,UPI,DELIVERED,App,26
3,ORD000550,2026-09-08 12:31:38,C00341,S01,P0128,1,211.0,20,168.8,UPI,DELIVERED,App,18
4,ORD001200,2026-09-16 12:51:37,C00316,S02,P0149,1,144.0,20,115.2,Card,DELIVERED,App,25



Last 5 rows:


,order_id,order_ts,customer_id,store_id,product_id,quantity,unit_price,discount_pct,order_amount,payment_mode,order_status,channel,delivery_minutes
2407,ORD001449,2026-09-19 13:22:42,C00209,S02,P0042,5,432.0,20,1728.0,UPI,DELIVERED,App,21
2408,ORD001950,2026-09-25 13:49:51,C00197,S13,P0061,1,819.0,0,819.0,Wallet,DELIVERED,Web,9
2409,ORD000347,2026-09-05 20:22:11,C00227,S12,P0056,2,663.0,15,1127.1,UPI,DELIVERED,App,24
2410,ORD000567,2026-09-08 17:21:34,C00012,S11,P0048,1,159.0,20,127.2,UPI,CANCELLED,App,
2411,ORD001203,2026-09-16 13:42:28,C00233,S03,P0077,3,369.0,10,996.3,UPI,DELIVERED,Web,31



5 random rows with fixed random_state=42:


,order_id,order_ts,customer_id,store_id,product_id,quantity,unit_price,discount_pct,order_amount,payment_mode,order_status,channel,delivery_minutes
1621,ORD000251,2026-09-04 14:03:20,C00209,S11,P0119,2,515.0,0,1030.0,UPI,DELIVERED,Web,16
2200,ORD001102,2026-09-15 08:18:24,C00125,S10,P0019,3,220.0,10,594.0,UPI,DELIVERED,Web,23
2399,ORD000050,2026-09-01 17:14:46,C00399,S13,P0072,1,396.0,0,396.0,Card,DELIVERED,App,30
1953,ORD002126,2026-09-27 15:52:57,C00360,S00,P0034,1,205.0,0,205.0,Wallet,DELIVERED,App,29
1798,ORD001170,2026-09-15 23:41:03,C00350,S06,P0089,1,492.0,15,418.2,UPI,DELIVERED,App,26


In [7]:
print("Shape:", orders_df.shape)
print("\nColumn names in source order:")
print(orders_df.columns.tolist())

print("\nPandas-inferred data types after conservative loading:")
print(orders_df.dtypes)

print("\nFirst 50 rows:")
display(orders_df.head(50))

print("\nLast 50 rows:")
display(orders_df.tail(50))

print("\n50 random rows with fixed random_state=42:")
display(orders_df.sample(n=50, random_state=42))

Shape: (2412, 13)

Column names in source order:
['order_id', 'order_ts', 'customer_id', 'store_id', 'product_id', 'quantity', 'unit_price', 'discount_pct', 'order_amount', 'payment_mode', 'order_status', 'channel', 'delivery_minutes']

Pandas-inferred data types after conservative loading:
order_id            str
order_ts            str
customer_id         str
store_id            str
product_id          str
quantity            str
unit_price          str
discount_pct        str
order_amount        str
payment_mode        str
order_status        str
channel             str
delivery_minutes    str
dtype: object

First 50 rows:


,order_id,order_ts,customer_id,store_id,product_id,quantity,unit_price,discount_pct,order_amount,payment_mode,order_status,channel,delivery_minutes
0,ORD001913,2026-09-24 22:56:41,C00005,S10,P0023,2,222.0,5,421.8,UPI,DELIVERED,App,35
1,ORD001791,2026-09-23 17:54:42,C00369,S05,P0136,1,360.0,0,360.0,UPI,DELIVERED,App,31
2,ORD002203,2026-09-28 15:53:19,C00010,S08,P0135,1,248.0,0,248.0,UPI,DELIVERED,App,26
3,ORD000550,2026-09-08 12:31:38,C00341,S01,P0128,1,211.0,20,168.8,UPI,DELIVERED,App,18
4,ORD001200,2026-09-16 12:51:37,C00316,S02,P0149,1,144.0,20,115.2,Card,DELIVERED,App,25
5,ORD001462,2026-09-19 15:32:55,C00265,S05,P0129,3,666.0,15,1698.3,Card,DELIVERED,App,8
6,ORD001314,2026-09-17 19:28:23,C00052,S02,P0018,3,186.0,0,558.0,UPI,DELIVERED,App,9
7,ORD000385,2026-09-06 10:02:54,C00136,S07,P0064,1,620.0,5,589.0,UPI,DELIVERED,App,17
8,ORD002073,2026-09-26 19:28:54,C00077,S12,P0082,1,338.0,10,304.2,Cash on Delivery,DELIVERED,App,17
9,ORD000787,2026-09-11 13:55:37,C00194,S03,P0034,1,205.0,0,205.0,Card,DELIVERED,App,29



Last 50 rows:


,order_id,order_ts,customer_id,store_id,product_id,quantity,unit_price,discount_pct,order_amount,payment_mode,order_status,channel,delivery_minutes
2362,ORD001837,2026-09-23 23:50:59,C00324,S12,P0146,1,337.0,0,337.0,UPI,DELIVERED,Web,32
2363,ORD001384,2026-09-18 18:42:03,C00009,S06,P0023,-1,222.0,15,-188.7,UPI,DELIVERED,App,32
2364,ORD001163,2026-09-15 22:13:04,C00182,S13,P0006,1,197.0,0,197.0,Wallet,DELIVERED,App,23
2365,ORD000514,2026-09-07 23:56:23,C00361,S13,P0017,1,120.0,10,108.0,Card,DELIVERED,App,29
2366,ORD000424,2026-09-06 19:16:13,C00148,S12,P0095,1,354.0,15,300.9,UPI,DELIVERED,App,480
2367,ORD002279,2026-09-29 16:50:50,C00214,S13,P0145,1,315.0,0,315.0,Cash on Delivery,DELIVERED,Web,10
2368,ORD001906,2026-09-24 21:55:25,C00313,S03,P0143,2,455.0,15,773.5,UPI,DELIVERED,App,8
2369,ORD000053,2026-09-01 17:39:43,C00314,S13,P0097,3,302.0,15,770.1,Cash on Delivery,DELIVERED,App,29
2370,ORD001823,2026-09-23 22:16:19,C00088,S04,P0057,1,820.0,0,820.0,UPI,DELIVERED,App,9
2371,ORD000371,2026-09-06 07:21:15,C00110,S08,P0120,3,344.0,5,980.4,Card,DELIVERED,App,12



50 random rows with fixed random_state=42:


,order_id,order_ts,customer_id,store_id,product_id,quantity,unit_price,discount_pct,order_amount,payment_mode,order_status,channel,delivery_minutes
1621,ORD000251,2026-09-04 14:03:20,C00209,S11,P0119,2,515.0,0,1030.0,UPI,DELIVERED,Web,16
2200,ORD001102,2026-09-15 08:18:24,C00125,S10,P0019,3,220.0,10,594.0,UPI,DELIVERED,Web,23
2399,ORD000050,2026-09-01 17:14:46,C00399,S13,P0072,1,396.0,0,396.0,Card,DELIVERED,App,30
1953,ORD002126,2026-09-27 15:52:57,C00360,S00,P0034,1,205.0,0,205.0,Wallet,DELIVERED,App,29
1798,ORD001170,2026-09-15 23:41:03,C00350,S06,P0089,1,492.0,15,418.2,UPI,DELIVERED,App,26
1034,ORD000556,2026-09-08 15:00:02,C00226,S08,P0027,2,448.0,15,761.6,UPI,DELIVERED,App,10
1866,ORD001884,2026-09-24 16:55:01,C00098,S05,P0086,5,23.0,10,103.5,UPI,DELIVERED,App,26
1766,ORD000750,2026-09-10 21:30:19,C00040,S09,P0025,1,238.0,0,238.0,Card,CANCELLED,App,
1656,ORD000210,2026-09-03 17:37:04,C00244,S01,P0135,1,248.0,0,248.0,UPI,DELIVERED,Web,20
756,ORD001433,2026-09-19 10:34:51,C00266,S08,P0121,2,334.0,20,534.4,UPI,DELIVERED,App,10


## Dataset inventory

| file_name | dataframe_name | rows | columns | duplicate_full_row_count | estimated_memory_bytes |
|---|---|---:|---:|---:|---:|
| customers.csv | customers_df | 405 | 10 | 0 | 237533 |
| orders.csv | orders_df | 2412 | 13 | 12 | 1713791 |
| payments.csv | payments_df | 2228 | 7 | 4 | 910980 |
| products.csv | products_df | 152 | 10 | 0 | 87582 |
| stores.csv | stores_df | 15 | 8 | 0 | 7038 |
| orders_2026-10-01_increment.csv | orders_increment_df | 160 | 13 | 0 | 114113 |

## orders.csv schema

**Shape:** `(2412, 13)`

**Column names in source order:**

```python
['order_id', 'order_ts', 'customer_id', 'store_id', 'product_id',
 'quantity', 'unit_price', 'discount_pct', 'order_amount',
 'payment_mode', 'order_status', 'channel', 'delivery_minutes']
```

**Inferred data types after conservative loading:**

```text
order_id            str
order_ts            str
customer_id         str
store_id            str
product_id          str
quantity            str
unit_price          str
discount_pct        str
order_amount        str
payment_mode        str
order_status        str
channel             str
delivery_minutes    str
```

## Loading warnings or errors

No CSV loading warnings or errors occurred. All six files loaded successfully, and all six DataFrames contained at least one row.

## My first observations

1. `order_ts` is of type `str` instead of a date-time type.
2. Cancelled order `ORD000682` has a payment despite the assignment instructions stating that cancelled orders have no payment.
3. `channel` can be either `App` or `Web` in the inspected data.
4. A negative quantity is present in the orders data.
5. The inventory shows 12 exact duplicate full rows in `orders.csv` and 4 exact duplicate full rows in `payments.csv`.
## Things I don't understand

1. I do not understand how to load these CSV files into DuckDB or how DuckDB decides the data type during loading.
2. I am unsure whether I need to create validation rules for every column, or whether basic validation rules are sufficient for this assignment.
3. I am unsure whether invalid records should be moved to a quarantine table or database instead of being deleted.
4. I am unsure which issues should be corrected automatically and which should only be reported.
5. I am unsure how to handle exact duplicate rows without accidentally removing legitimate repeated transactions.
6. I am unsure whether the incremental orders file should be validated separately or combined with `orders.csv` first.
7. I am unsure how to validate relationships between orders, customers, products, stores, and payments.

# Stage 2: Structural Profiling

This stage measures missing values, exact duplicates, candidate-key repetition, and source-schema alignment. It does not modify, clean, standardise, impute, delete, or classify any source values.

## 1. Missing-value profiling

This section counts actual nulls and blank strings separately for every column in every DataFrame. It matters because missingness can affect later analysis, while keeping the counts separate preserves the distinction between a null value and an empty text value. These results do not prove that non-missing values are valid or meaningful.

In [8]:
missing_profile_rows = []

for dataframe_name, dataframe in dataframes.items():
    for column_name in dataframe.columns:
        null_count = int(dataframe[column_name].isna().sum())
        blank_count = int(
            dataframe[column_name]
            .astype("string")
            .str.strip()
            .eq("")
            .sum()
        )
        combined_missing_count = null_count + blank_count
        missing_profile_rows.append(
            {
                "dataset": dataframe_name,
                "column": column_name,
                "rows_checked": len(dataframe),
                "null_count": null_count,
                "blank_string_count": blank_count,
                "combined_missing_count": combined_missing_count,
                "combined_missing_percentage": (
                    combined_missing_count / len(dataframe) * 100
                ),
            }
        )

missing_profile_df = pd.DataFrame(missing_profile_rows)

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.width", None,
):
    display(missing_profile_df)

,dataset,column,rows_checked,null_count,blank_string_count,combined_missing_count,combined_missing_percentage
0,customers_df,customer_id,405,0,0,0,0.000000
1,customers_df,full_name,405,0,0,0,0.000000
2,customers_df,email,405,0,0,0,0.000000
3,customers_df,phone,405,0,0,0,0.000000
4,customers_df,city,405,0,6,6,1.481481
5,customers_df,area,405,0,0,0,0.000000
6,customers_df,pincode,405,0,0,0,0.000000
7,customers_df,signup_date,405,0,0,0,0.000000
8,customers_df,loyalty_tier,405,0,0,0,0.000000
9,customers_df,is_active,405,0,0,0,0.000000


## 2. Exact full-row duplicate profiling

This section measures duplicate occurrences, all rows belonging to duplicate groups, and distinct repeated full-row values. It matters because repeated source records can affect counts, but these measurements do not decide whether a duplicate is legitimate or erroneous.

In [9]:
duplicate_profile_rows = []

for dataframe_name, dataframe in dataframes.items():
    duplicate_occurrences = int(dataframe.duplicated().sum())
    duplicate_group_mask = dataframe.duplicated(keep=False)
    duplicate_group_rows = int(duplicate_group_mask.sum())
    duplicate_groups = int(
        dataframe.loc[duplicate_group_mask].drop_duplicates().shape[0]
    )
    duplicate_profile_rows.append(
        {
            "dataset": dataframe_name,
            "rows_checked": len(dataframe),
            "duplicate_occurrences": duplicate_occurrences,
            "rows_in_duplicate_groups": duplicate_group_rows,
            "distinct_duplicate_groups": duplicate_groups,
        }
    )

duplicate_profile_df = pd.DataFrame(duplicate_profile_rows)
duplicate_profile_df

,dataset,rows_checked,duplicate_occurrences,rows_in_duplicate_groups,distinct_duplicate_groups
0,customers_df,405,0,0,0
1,orders_df,2412,12,24,12
2,payments_df,2228,4,8,4
3,products_df,152,0,0,0
4,stores_df,15,0,0,0
5,orders_increment_df,160,0,0,0


### Duplicate groups in `orders_df` and `payments_df`

This section displays every row in the exact duplicate groups, sorted by all source columns so identical rows are adjacent. It matters because it makes the evidence inspectable; it does not remove rows or prove that the repeated rows should be discarded.

In [10]:
orders_duplicate_mask = orders_df.duplicated(keep=False)
orders_duplicate_rows = orders_df.loc[orders_duplicate_mask].sort_values(
    by=list(orders_df.columns),
    kind="stable",
)

payments_duplicate_mask = payments_df.duplicated(keep=False)
payments_duplicate_rows = payments_df.loc[payments_duplicate_mask].sort_values(
    by=list(payments_df.columns),
    kind="stable",
)

print("All rows in exact duplicate groups from orders_df:")
display(orders_duplicate_rows)

print("All rows in exact duplicate groups from payments_df:")
display(payments_duplicate_rows)

assert orders_duplicate_rows.equals(
    orders_df.loc[orders_duplicate_rows.index]
    .sort_values(by=list(orders_df.columns), kind="stable")
)
assert payments_duplicate_rows.equals(
    payments_df.loc[payments_duplicate_rows.index]
    .sort_values(by=list(payments_df.columns), kind="stable")
)
assert orders_duplicate_rows.index.isin(orders_df.index).all()
assert payments_duplicate_rows.index.isin(payments_df.index).all()

All rows in exact duplicate groups from orders_df:


,order_id,order_ts,customer_id,store_id,product_id,quantity,unit_price,discount_pct,order_amount,payment_mode,order_status,channel,delivery_minutes
2027,ORD000539,2026-09-08 10:36:28,C00102,S13,P0001,3,107.0,10,288.9,UPI,DELIVERED,App,13
2144,ORD000539,2026-09-08 10:36:28,C00102,S13,P0001,3,107.0,10,288.9,UPI,DELIVERED,App,13
444,ORD000783,2026-09-11 12:43:06,C00267,S13,P0110,4,42.0,10,151.2,Card,DELIVERED,App,32
2086,ORD000783,2026-09-11 12:43:06,C00267,S13,P0110,4,42.0,10,151.2,Card,DELIVERED,App,32
1769,ORD001110,2026-09-15 09:23:24,C00329,S13,P0068,3,649.0,5,1849.65,UPI,DELIVERED,App,9
2393,ORD001110,2026-09-15 09:23:24,C00329,S13,P0068,3,649.0,5,1849.65,UPI,DELIVERED,App,9
732,ORD001239,2026-09-16 21:34:44,C00355,S10,P0042,5,432.0,5,2052.0,UPI,DELIVERED,App,33
2057,ORD001239,2026-09-16 21:34:44,C00355,S10,P0042,5,432.0,5,2052.0,UPI,DELIVERED,App,33
237,ORD001326,2026-09-17 22:32:55,C00379,S12,P0146,1,337.0,15,286.45,Card,DELIVERED,App,24
2202,ORD001326,2026-09-17 22:32:55,C00379,S12,P0146,1,337.0,15,286.45,Card,DELIVERED,App,24


All rows in exact duplicate groups from payments_df:


,payment_id,order_id,payment_ts,payment_mode,amount_paid,payment_status,gateway_ref
261,PAY0000509,ORD002122,2026-09-27 14:57:42,UPI,148.8,SUCCESS,GW5719011844
1030,PAY0000509,ORD002122,2026-09-27 14:57:42,UPI,148.8,SUCCESS,GW5719011844
145,PAY0000755,ORD000094,2026-09-02 10:58:26,Card,403.2,SUCCESS,GW2899060380
2117,PAY0000755,ORD000094,2026-09-02 10:58:26,Card,403.2,SUCCESS,GW2899060380
19,PAY0001651,ORD000188,2026-09-03 13:23:25,Card,94.5,SUCCESS,GW7909341632
1853,PAY0001651,ORD000188,2026-09-03 13:23:25,Card,94.5,SUCCESS,GW7909341632
1346,PAY0002000,ORD000513,2026-09-07 23:16:17,Wallet,684.0,SUCCESS,GW2663042309
1875,PAY0002000,ORD000513,2026-09-07 23:16:17,Wallet,684.0,SUCCESS,GW2663042309


## 3. Candidate-key profiling

This section profiles the proposed identifier column in each DataFrame. It matters because missing or repeated identifiers can affect joins and counting, but repeated values are evidence for investigation, not proof of an error.

In [11]:
candidate_keys = {
    "customers_df": "customer_id",
    "products_df": "product_id",
    "stores_df": "store_id",
    "orders_df": "order_id",
    "payments_df": "payment_id",
    "orders_increment_df": "order_id",
}

candidate_key_profile_rows = []

for dataframe_name, key_column in candidate_keys.items():
    dataframe = dataframes[dataframe_name]
    key_values = dataframe[key_column]
    missing_key_mask = key_values.isna() | key_values.astype("string").str.strip().eq("")
    non_missing_keys = key_values.loc[~missing_key_mask]
    repeated_key_mask = (
        key_values.astype("string").duplicated(keep=False) & ~missing_key_mask
    )
    candidate_key_profile_rows.append(
        {
            "dataset": dataframe_name,
            "key_column": key_column,
            "rows_checked": len(dataframe),
            "missing_key_count": int(missing_key_mask.sum()),
            "unique_non_missing_key_count": int(non_missing_keys.nunique()),
            "duplicate_occurrences_after_first": int(
                non_missing_keys.duplicated().sum()
            ),
            "all_rows_in_repeated_key_groups": int(repeated_key_mask.sum()),
            "distinct_repeated_key_values": int(
                key_values.loc[repeated_key_mask].nunique()
            ),
        }
    )

candidate_key_profile_df = pd.DataFrame(candidate_key_profile_rows)
candidate_key_profile_df

,dataset,key_column,rows_checked,missing_key_count,unique_non_missing_key_count,duplicate_occurrences_after_first,all_rows_in_repeated_key_groups,distinct_repeated_key_values
0,customers_df,customer_id,405,0,400,5,10,5
1,products_df,product_id,152,0,150,2,4,2
2,stores_df,store_id,15,0,15,0,0,0
3,orders_df,order_id,2412,0,2400,12,24,12
4,payments_df,payment_id,2228,0,2224,4,8,4
5,orders_increment_df,order_id,160,0,160,0,0,0


## 4. Classify repeated keys for orders and payments

This section separates repeated-key groups whose full rows are identical from repeated-key groups with different non-key values. It matters because those patterns provide different evidence for follow-up, but neither pattern is assumed to be an error here.

In [12]:
def split_repeated_key_groups(dataframe, key_column):
    key_values = dataframe[key_column].astype("string")
    missing_key_mask = dataframe[key_column].isna() | key_values.str.strip().eq("")
    repeated_key_mask = key_values.duplicated(keep=False) & ~missing_key_mask
    repeated_rows = dataframe.loc[repeated_key_mask]

    exact_duplicate_group_rows = repeated_rows.groupby(
        key_column,
        sort=False,
        dropna=False,
    ).filter(lambda group: len(group.drop_duplicates()) == 1)
    non_identical_repeated_key_rows = repeated_rows.groupby(
        key_column,
        sort=False,
        dropna=False,
    ).filter(lambda group: len(group.drop_duplicates()) > 1)
    return exact_duplicate_group_rows, non_identical_repeated_key_rows

orders_exact_key_duplicate_rows, orders_non_identical_key_rows = (
    split_repeated_key_groups(orders_df, "order_id")
)
payments_exact_key_duplicate_rows, payments_non_identical_key_rows = (
    split_repeated_key_groups(payments_df, "payment_id")
)

print("Orders: repeated keys with identical full rows")
display(orders_exact_key_duplicate_rows)
print("Orders: repeated keys with differing non-key values")
display(orders_non_identical_key_rows)

print("Payments: repeated keys with identical full rows")
display(payments_exact_key_duplicate_rows)
print("Payments: repeated keys with differing non-key values")
display(payments_non_identical_key_rows)

for subset, source in [
    (orders_exact_key_duplicate_rows, orders_df),
    (orders_non_identical_key_rows, orders_df),
    (payments_exact_key_duplicate_rows, payments_df),
    (payments_non_identical_key_rows, payments_df),
]:
    assert subset.index.isin(source.index).all()
    assert subset.equals(source.loc[subset.index])

Orders: repeated keys with identical full rows


,order_id,order_ts,customer_id,store_id,product_id,quantity,unit_price,discount_pct,order_amount,payment_mode,order_status,channel,delivery_minutes
5,ORD001462,2026-09-19 15:32:55,C00265,S05,P0129,3,666.0,15,1698.3,Card,DELIVERED,App,8
237,ORD001326,2026-09-17 22:32:55,C00379,S12,P0146,1,337.0,15,286.45,Card,DELIVERED,App,24
269,ORD001462,2026-09-19 15:32:55,C00265,S05,P0129,3,666.0,15,1698.3,Card,DELIVERED,App,8
355,ORD001434,2026-09-19 10:55:15,C00307,S11,P0019,1,220.0,0,220.0,Card,DELIVERED,Web,20
444,ORD000783,2026-09-11 12:43:06,C00267,S13,P0110,4,42.0,10,151.2,Card,DELIVERED,App,32
732,ORD001239,2026-09-16 21:34:44,C00355,S10,P0042,5,432.0,5,2052.0,UPI,DELIVERED,App,33
780,ORD001692,2026-09-22 10:27:56,C00032,S10,P0145,1,315.0,15,267.75,UPI,DELIVERED,Web,29
1023,ORD001870,2026-09-24 14:09:09,C00113,S01,P0051,1,283.0,5,268.85,UPI,DELIVERED,App,12
1113,ORD001441,2026-09-19 12:15:53,C00206,S02,P0090,1,527.0,0,527.0,Card,DELIVERED,App,29
1119,ORD001441,2026-09-19 12:15:53,C00206,S02,P0090,1,527.0,0,527.0,Card,DELIVERED,App,29


Orders: repeated keys with differing non-key values


,order_id,order_ts,customer_id,store_id,product_id,quantity,unit_price,discount_pct,order_amount,payment_mode,order_status,channel,delivery_minutes


Payments: repeated keys with identical full rows


,payment_id,order_id,payment_ts,payment_mode,amount_paid,payment_status,gateway_ref
19,PAY0001651,ORD000188,2026-09-03 13:23:25,Card,94.5,SUCCESS,GW7909341632
145,PAY0000755,ORD000094,2026-09-02 10:58:26,Card,403.2,SUCCESS,GW2899060380
261,PAY0000509,ORD002122,2026-09-27 14:57:42,UPI,148.8,SUCCESS,GW5719011844
1030,PAY0000509,ORD002122,2026-09-27 14:57:42,UPI,148.8,SUCCESS,GW5719011844
1346,PAY0002000,ORD000513,2026-09-07 23:16:17,Wallet,684.0,SUCCESS,GW2663042309
1853,PAY0001651,ORD000188,2026-09-03 13:23:25,Card,94.5,SUCCESS,GW7909341632
1875,PAY0002000,ORD000513,2026-09-07 23:16:17,Wallet,684.0,SUCCESS,GW2663042309
2117,PAY0000755,ORD000094,2026-09-02 10:58:26,Card,403.2,SUCCESS,GW2899060380


Payments: repeated keys with differing non-key values


,payment_id,order_id,payment_ts,payment_mode,amount_paid,payment_status,gateway_ref


## 5. Source schema comparison

This section compares the column names and their order in the main orders file and the increment file. It matters because downstream loading depends on a compatible source layout; a matching schema does not prove that the values are valid or that the files contain the same records.

In [13]:
orders_columns = orders_df.columns.tolist()
increment_columns = orders_increment_df.columns.tolist()
common_order_in_orders = [
    column for column in orders_columns if column in increment_columns
]
common_order_in_increment = [
    column for column in increment_columns if column in orders_columns
]

missing_from_increment = [
    column for column in orders_columns if column not in increment_columns
]
additional_in_increment = [
    column for column in increment_columns if column not in orders_columns
]
reordered_columns = (
    {
        "orders_order": common_order_in_orders,
        "increment_order": common_order_in_increment,
    }
    if common_order_in_orders != common_order_in_increment
    else {}
)
schema_matches_exactly = orders_columns == increment_columns

schema_comparison_df = pd.DataFrame(
    {
        "comparison": [
            "orders columns",
            "increment columns",
            "schemas match exactly",
            "missing from increment",
            "additional in increment",
            "reordered columns",
        ],
        "result": [
            orders_columns,
            increment_columns,
            schema_matches_exactly,
            missing_from_increment,
            additional_in_increment,
            reordered_columns,
        ],
    }
)

schema_comparison_df

,comparison,result
0,orders columns,"[order_id, order_ts, customer_id, store_id, pr..."
1,increment columns,"[order_id, order_ts, customer_id, store_id, pr..."
2,schemas match exactly,True
3,missing from increment,[]
4,additional in increment,[]
5,reordered columns,{}


## 6. Evidence summary

This section combines the structural checks into one table with a consistent status. It matters because a single view makes follow-up easier; `INVESTIGATE` means the check found something to inspect, not that the data is definitely wrong.

In [14]:
evidence_rows = []

for dataframe_name, dataframe in dataframes.items():
    dataframe_missing = missing_profile_df.loc[
        missing_profile_df["dataset"] == dataframe_name
    ]
    missing_rows_failed = int(
        dataframe.apply(
            lambda row: any(
                value is None or (isinstance(value, str) and value.strip() == "")
                for value in row
            ),
            axis=1,
        ).sum()
    )
    evidence_rows.append(
        {
            "dataset": dataframe_name,
            "check_name": "missing values or blank strings",
            "rows_checked": len(dataframe),
            "rows_failed": missing_rows_failed,
            "distinct_keys_failed": 0,
            "status": "PASS" if missing_rows_failed == 0 else "INVESTIGATE",
        }
    )

    duplicate_result = duplicate_profile_df.loc[
        duplicate_profile_df["dataset"] == dataframe_name
    ].iloc[0]
    duplicate_rows_failed = int(duplicate_result["duplicate_occurrences"])
    evidence_rows.append(
        {
            "dataset": dataframe_name,
            "check_name": "exact full-row duplicates",
            "rows_checked": len(dataframe),
            "rows_failed": duplicate_rows_failed,
            "distinct_keys_failed": int(duplicate_result["distinct_duplicate_groups"]),
            "status": (
                "PASS" if duplicate_rows_failed == 0 else "INVESTIGATE"
            ),
        }
    )

for dataframe_name, key_column in candidate_keys.items():
    key_result = candidate_key_profile_df.loc[
        candidate_key_profile_df["dataset"] == dataframe_name
    ].iloc[0]
    key_rows_failed = int(
        key_result["missing_key_count"]
        + key_result["duplicate_occurrences_after_first"]
    )
    evidence_rows.append(
        {
            "dataset": dataframe_name,
            "check_name": f"candidate key: {key_column}",
            "rows_checked": len(dataframes[dataframe_name]),
            "rows_failed": key_rows_failed,
            "distinct_keys_failed": int(key_result["distinct_repeated_key_values"]),
            "status": "PASS" if key_rows_failed == 0 else "INVESTIGATE",
        }
    )

schema_rows_failed = 0 if schema_matches_exactly else len(orders_df)
evidence_rows.append(
    {
        "dataset": "orders_df and orders_increment_df",
        "check_name": "source schema comparison",
        "rows_checked": len(orders_df),
        "rows_failed": schema_rows_failed,
        "distinct_keys_failed": 0,
        "status": "PASS" if schema_rows_failed == 0 else "INVESTIGATE",
    }
)

evidence_summary_df = pd.DataFrame(
    evidence_rows,
    columns=[
        "dataset",
        "check_name",
        "rows_checked",
        "rows_failed",
        "distinct_keys_failed",
        "status",
    ],
)
evidence_summary_df

,dataset,check_name,rows_checked,rows_failed,distinct_keys_failed,status
0,customers_df,missing values or blank strings,405,6,0,INVESTIGATE
1,customers_df,exact full-row duplicates,405,0,0,PASS
2,orders_df,missing values or blank strings,2412,182,0,INVESTIGATE
3,orders_df,exact full-row duplicates,2412,12,12,INVESTIGATE
4,payments_df,missing values or blank strings,2228,301,0,INVESTIGATE
5,payments_df,exact full-row duplicates,2228,4,4,INVESTIGATE
6,products_df,missing values or blank strings,152,3,0,INVESTIGATE
7,products_df,exact full-row duplicates,152,0,0,PASS
8,stores_df,missing values or blank strings,15,1,0,INVESTIGATE
9,stores_df,exact full-row duplicates,15,0,0,PASS


## 7. Structural profiling assertions

These assertions verify the bookkeeping behind the reports: every check covers the full source DataFrame, and every displayed duplicate subset is sourced from its original DataFrame. They do not assert that any quality check passes.

In [15]:
for dataframe_name, dataframe in dataframes.items():
    missing_rows = missing_profile_df.loc[
        missing_profile_df["dataset"] == dataframe_name
    ]
    assert (missing_rows["rows_checked"] == len(dataframe)).all()

    duplicate_rows = duplicate_profile_df.loc[
        duplicate_profile_df["dataset"] == dataframe_name
    ]
    assert (duplicate_rows["rows_checked"] == len(dataframe)).all()

    key_rows = candidate_key_profile_df.loc[
        candidate_key_profile_df["dataset"] == dataframe_name
    ]
    assert (key_rows["rows_checked"] == len(dataframe)).all()

assert (
    evidence_summary_df.loc[
        evidence_summary_df["dataset"] != "orders_df and orders_increment_df",
        "rows_checked",
    ]
    .map(lambda rows_checked: rows_checked in [len(dataframe) for dataframe in dataframes.values()])
    .all()
)
assert evidence_summary_df.loc[
    evidence_summary_df["dataset"] == "orders_df and orders_increment_df",
    "rows_checked",
].eq(len(orders_df)).all()

assert orders_duplicate_rows.equals(
    orders_df.loc[orders_duplicate_rows.index]
    .sort_values(by=list(orders_df.columns), kind="stable")
)
assert payments_duplicate_rows.equals(
    payments_df.loc[payments_duplicate_rows.index]
    .sort_values(by=list(payments_df.columns), kind="stable")
)
assert orders_duplicate_rows.index.isin(orders_df.index).all()
assert payments_duplicate_rows.index.isin(payments_df.index).all()

print("Structural profiling assertions passed without asserting that checks pass.")

Structural profiling assertions passed without asserting that checks pass.


## Stage 2 observations and questions

## Missing-value summary

Only columns where the combined missing count is greater than zero are shown. The `null_count` column counts actual null values, while `blank_string_count` counts empty or whitespace-only strings.

| dataset | column | rows_checked | null_count | blank_string_count | combined_missing_count | combined_missing_percentage |
|---|---|---:|---:|---:|---:|---:|
| customers_df | city | 405 | 0 | 6 | 6 | 1.48% |
| orders_df | order_ts | 2412 | 0 | 1 | 1 | 0.04% |
| orders_df | customer_id | 2412 | 0 | 18 | 18 | 0.75% |
| orders_df | delivery_minutes | 2412 | 0 | 164 | 164 | 6.80% |
| payments_df | gateway_ref | 2228 | 0 | 301 | 301 | 13.51% |
| products_df | category | 152 | 0 | 3 | 3 | 1.97% |
| stores_df | zone | 15 | 0 | 1 | 1 | 6.67% |

`orders_increment_df` has no columns with a combined missing count greater than zero.

## Candidate-key summary

| dataset | key column | rows checked | missing key count | unique non-missing key count | duplicate occurrences after first | all rows in repeated-key groups | distinct repeated key values |
|---|---|---:|---:|---:|---:|---:|---:|
| customers_df | customer_id | 405 | 0 | 400 | 5 | 10 | 5 |
| products_df | product_id | 152 | 0 | 150 | 2 | 4 | 2 |
| stores_df | store_id | 15 | 0 | 15 | 0 | 0 | 0 |
| orders_df | order_id | 2412 | 0 | 2400 | 12 | 24 | 12 |
| payments_df | payment_id | 2228 | 0 | 2224 | 4 | 8 | 4 |
| orders_increment_df | order_id | 160 | 0 | 160 | 0 | 0 | 0 |

## Exact duplicates

| dataset | duplicate occurrences | total involved rows | distinct groups |
|---|---:|---:|---:|
| orders | 12 | 24 | 12 |
| payments | 4 | 8 | 4 |

No rows were removed. The duplicate groups remain available in the displayed profiling outputs.

## Repeated non-identical keys

| dataset | rows in repeated non-identical-key groups | distinct repeated key values | examples |
|---|---:|---:|---|
| orders | 0 | 0 | None; all repeated order IDs belong to exact full-row duplicate groups. |
| payments | 0 | 0 | None; all repeated payment IDs belong to exact full-row duplicate groups. |

## Schema comparison

The increment schema matches `orders_df` exactly, including column names and column order.

- Schemas match exactly: `True`
- Missing from increment: none
- Additional in increment: none
- Reordered columns: none

## My interpretation

I see no actual null values in the given dataset. However, the structural profiling shows blank strings in several columns, so “no nulls” does not mean “no missing-looking values.”

No other interpretations so far.

## Questions

I think we are overdoing this data profiling task.


# Stage 3: Column Profiling

This stage profiles numeric, date, and categorical columns using derived values only. It does not clean, overwrite, quarantine, or classify any source values, and it does not apply business rules.

## 1. Numeric column profiling

This section attempts numeric parsing in temporary Series and reports the valid minimum, valid maximum, and non-blank values that could not be parsed. It is useful for seeing the numeric shape of each column; it does not correct values, and it does not decide whether a numeric value is sensible for the business.

In [16]:
numeric_columns = {
    "customers_df": [],
    "orders_df": [
        "quantity",
        "unit_price",
        "discount_pct",
        "order_amount",
        "delivery_minutes",
    ],
    "payments_df": ["amount_paid"],
    "products_df": ["mrp", "cost_price"],
    "stores_df": ["capacity_orders_per_day"],
    "orders_increment_df": [
        "quantity",
        "unit_price",
        "discount_pct",
        "order_amount",
        "delivery_minutes",
    ],
}

numeric_profile_rows = []
for dataframe_name, column_names in numeric_columns.items():
    dataframe = dataframes[dataframe_name]
    for column_name in column_names:
        source_values = dataframe[column_name].astype("string")
        non_blank_mask = source_values.notna() & source_values.str.strip().ne("")
        parsed_values = pd.to_numeric(source_values, errors="coerce")
        invalid_mask = non_blank_mask & parsed_values.isna()
        valid_values = parsed_values.loc[non_blank_mask & parsed_values.notna()]
        numeric_profile_rows.append(
            {
                "dataset": dataframe_name,
                "column": column_name,
                "rows_checked": len(dataframe),
                "min": valid_values.min() if not valid_values.empty else pd.NA,
                "max": valid_values.max() if not valid_values.empty else pd.NA,
                "invalid_numeric_count": int(invalid_mask.sum()),
            }
        )

numeric_profile_df = pd.DataFrame(numeric_profile_rows)
numeric_profile_df

,dataset,column,rows_checked,min,max,invalid_numeric_count
0,orders_df,quantity,2412,-3.00,5.00,0
1,orders_df,unit_price,2412,10.00,883.00,0
2,orders_df,discount_pct,2412,0.00,250.00,0
3,orders_df,order_amount,2412,-1593.00,9576.00,0
4,orders_df,delivery_minutes,2412,8.00,1440.00,0
5,payments_df,amount_paid,2228,-1593.00,9576.00,0
6,products_df,mrp,152,-203.00,883.00,0
7,products_df,cost_price,152,6.79,631.79,0
8,stores_df,capacity_orders_per_day,15,0.00,600.00,0
9,orders_increment_df,quantity,160,1.00,4.00,0


## 2. Date column profiling

This section parses date columns in temporary Series and reports the valid date range, non-blank parse failures, and dates after the fixed reference date. It is useful for understanding date coverage and parseability; it does not decide whether a date is realistic or appropriate for the business.

In [17]:
date_columns = {
    "customers_df": ["signup_date"],
    "orders_df": ["order_ts"],
    "payments_df": ["payment_ts"],
    "products_df": ["launch_date"],
    "stores_df": ["open_date"],
    "orders_increment_df": ["order_ts"],
}

future_reference_date = pd.Timestamp("2026-10-14")
date_profile_rows = []

for dataframe_name, column_names in date_columns.items():
    dataframe = dataframes[dataframe_name]
    for column_name in column_names:
        source_values = dataframe[column_name].astype("string")
        non_blank_mask = source_values.notna() & source_values.str.strip().ne("")
        parsed_dates = pd.to_datetime(
            source_values,
            errors="coerce",
            format="mixed",
        )
        invalid_mask = non_blank_mask & parsed_dates.isna()
        valid_dates = parsed_dates.loc[non_blank_mask & parsed_dates.notna()]
        date_profile_rows.append(
            {
                "dataset": dataframe_name,
                "column": column_name,
                "rows_checked": len(dataframe),
                "min_date": valid_dates.min() if not valid_dates.empty else pd.NaT,
                "max_date": valid_dates.max() if not valid_dates.empty else pd.NaT,
                "invalid_date_count": int(invalid_mask.sum()),
                "future_date_count": int(
                    (valid_dates > future_reference_date).sum()
                ),
            }
        )

date_profile_df = pd.DataFrame(date_profile_rows)
date_profile_df

,dataset,column,rows_checked,min_date,max_date,invalid_date_count,future_date_count
0,customers_df,signup_date,405,2024-06-01 00:00:00,2027-01-05 00:00:00,0,3
1,orders_df,order_ts,2412,2026-09-01 06:30:15,2027-01-01 09:00:00,5,7
2,payments_df,payment_ts,2228,2026-09-01 06:30:24,2027-01-01 09:01:46,0,7
3,products_df,launch_date,152,2023-01-03 00:00:00,2025-09-20 00:00:00,0,0
4,stores_df,open_date,15,2024-02-14 00:00:00,2025-07-28 00:00:00,0,0
5,orders_increment_df,order_ts,160,2026-09-01 09:38:13,2026-10-01 22:33:00,0,0


## 3. Categorical column profiling

This section reports distinct values, frequency counts, and values occurring fewer than five times for the explicitly selected categorical fields. It matters because distributions can reveal how values are represented; it does not determine whether uncommon or frequent values are correct.

In [18]:
categorical_columns = {
    "customers_df": ["city", "area", "loyalty_tier", "is_active"],
    "orders_df": ["payment_mode", "order_status", "channel"],
    "payments_df": ["payment_mode", "payment_status"],
    "products_df": [
        "category",
        "sub_category",
        "brand",
        "unit",
        "is_perishable",
    ],
    "stores_df": ["zone", "area"],
    "orders_increment_df": ["payment_mode", "order_status", "channel"],
}

categorical_summary_rows = []
categorical_frequency_rows = []

for dataframe_name, column_names in categorical_columns.items():
    dataframe = dataframes[dataframe_name]
    for column_name in column_names:
        value_counts = dataframe[column_name].value_counts(dropna=False)
        categorical_summary_rows.append(
            {
                "dataset": dataframe_name,
                "column": column_name,
                "rows_checked": len(dataframe),
                "distinct_value_count": int(value_counts.size),
                "distinct_values": value_counts.index.tolist(),
            }
        )
        for value, count in value_counts.items():
            categorical_frequency_rows.append(
                {
                    "dataset": dataframe_name,
                    "column": column_name,
                    "value": value,
                    "count": int(count),
                    "is_fewer_than_5": int(count) < 5,
                }
            )

categorical_summary_df = pd.DataFrame(categorical_summary_rows)
categorical_frequency_df = pd.DataFrame(categorical_frequency_rows)
categorical_rare_values_df = categorical_frequency_df.loc[
    categorical_frequency_df["is_fewer_than_5"]
].reset_index(drop=True)

print("Distinct values by categorical column:")
display(categorical_summary_df)

print("Frequency counts:")
display(categorical_frequency_df)

print("Values occurring fewer than 5 times:")
display(categorical_rare_values_df)

Distinct values by categorical column:


,dataset,column,rows_checked,distinct_value_count,distinct_values
0,customers_df,city,405,2,"[Bengaluru, ]"
1,customers_df,area,405,15,"[Indiranagar, RT Nagar, BTM Layout, Yelahanka,..."
2,customers_df,loyalty_tier,405,7,"[Bronze, Silver, Gold, Platinum, BRONZE, bronz..."
3,customers_df,is_active,405,2,"[Y, N]"
4,orders_df,payment_mode,2412,9,"[UPI, Card, Cash on Delivery, Wallet, upi, Cas..."
5,orders_df,order_status,2412,5,"[DELIVERED, CANCELLED, RETURNED, Pending??, de..."
6,orders_df,channel,2412,2,"[App, Web]"
7,payments_df,payment_mode,2228,9,"[UPI, Card, Cash on Delivery, Wallet, upi, UPI..."
8,payments_df,payment_status,2228,3,"[SUCCESS, REFUNDED, FAILED]"
9,products_df,category,152,7,"[Dairy & Bakery, Household, Fruits & Vegetable..."


Frequency counts:


,dataset,column,value,count,is_fewer_than_5
0,customers_df,city,Bengaluru,399,False
1,customers_df,city,,6,False
2,customers_df,area,Indiranagar,43,False
3,customers_df,area,RT Nagar,37,False
4,customers_df,area,BTM Layout,34,False
...,...,...,...,...,...
132,orders_increment_df,payment_mode,Cash on Delivery,40,False
133,orders_increment_df,payment_mode,Wallet,30,False
134,orders_increment_df,order_status,DELIVERED,150,False
135,orders_increment_df,order_status,RETURNED,10,False


Values occurring fewer than 5 times:


,dataset,column,value,count,is_fewer_than_5
0,customers_df,loyalty_tier,BRONZE,3,True
1,customers_df,loyalty_tier,bronze,3,True
2,customers_df,loyalty_tier,SILVER,2,True
3,orders_df,payment_mode,upi,4,True
4,orders_df,payment_mode,Cashh,2,True
5,orders_df,payment_mode,UPI,2,True
6,orders_df,payment_mode,CARD,1,True
7,orders_df,payment_mode,cod,1,True
8,orders_df,order_status,Pending??,3,True
9,orders_df,order_status,delivered,2,True


## Stage 3 observations and questions

### 1. Negative quantities are present, but numeric parsing does not call them invalid

**Observation:** `quantity` contains negative values even though every non-blank quantity is syntactically numeric.

**Evidence:** The numeric profile reports `orders_df.quantity` with a minimum of `-3` and `invalid_numeric_count = 0`. The same profile reports `orders_df.order_amount` with a minimum of `-1593.00`.

**Business impact:** If negative quantities or amounts enter revenue, inventory, or margin calculations without an agreed interpretation, totals could be understated or distorted.

**Question:** Why is a negative quantity not treated as invalid? Does it represent a return, a correction, or a source-system error? This needs a business decision before creating a validation rule.

### 2. Some numeric columns contain extreme values that are still syntactically valid

**Observation:** The parser accepts all non-blank numeric strings, including unusually large delivery and discount values.

**Evidence:** `delivery_minutes` ranges from `8` to `1440`, and `discount_pct` ranges from `0` to `250` in `orders_df`. Both have `invalid_numeric_count = 0`.

**Business impact:** These values could materially affect delivery-service metrics and revenue calculations if consumed without agreed business limits.

**Question:** What operational ranges should NammaMart approve for delivery time and discount percentage?

### 3. Several date fields contain invalid or future dates

**Observation:** Most date values parse, but some order timestamps are invalid or later than the fixed reference date of `2026-10-14`.

**Evidence:** `orders_df.order_ts` has `invalid_date_count = 5` and `future_date_count = 7`. `customers_df.signup_date` has `future_date_count = 3`, and `payments_df.payment_ts` has `future_date_count = 7`.

**Business impact:** Invalid or future timestamps can affect daily reporting, freshness checks, late-arriving data analysis, and reconciliation by date.

**Question:** Are future-dated records expected test data, corrections, or source-system timestamp problems?

### 4. Categorical values are represented inconsistently

**Observation:** The same apparent category appears with different casing or spelling in the source values.

**Evidence:** The categorical profile shows values such as `upi`, `UPI`, `CARD`, `cod`, and `Cashh` in payment modes. It also shows `Pending??` and `delivered` among order statuses, while the increment file contains only `DELIVERED` and `RETURNED`.

**Business impact:** Grouping revenue or payment totals by raw category values could split one business concept into multiple report groups or exclude records from expected groups.

**Question:** Which source values are approved representations, and which should be reported separately before any standardisation is designed?

### 5. The increment file is structurally cleaner than the historical orders file

**Observation:** The increment file has the same schema as `orders_df`, but its profiling results show no invalid numeric or date values and no repeated candidate order IDs.

**Evidence:** The schema comparison reports an exact match. `orders_increment_df` has zero invalid numeric values, zero invalid dates, zero future dates, and zero repeated `order_id` values in the Stage 3 outputs.

**Business impact:** Applying the same downstream assumptions to both files may produce different results because the historical and incremental feeds have different observed quality patterns.

**Question:** Should the two feeds use the same validation checks while keeping separate profiling and run-level evidence?


# Stage 4: Cross-column Business Rule Profiling

This stage compares related columns and reports business-rule evidence using derived Series only. It does not clean, modify, standardise, quarantine, delete, or classify source records.

## 1. Order amount formula validation

This check compares the source `order_amount` with a derived expected amount using quantity, unit price, and discount. It matters because revenue calculations depend on these columns agreeing; a mismatch is evidence to investigate, not an automatic correction.

In [19]:
rule_sample_rows = []

quantity_numeric = pd.to_numeric(orders_df["quantity"], errors="coerce")
unit_price_numeric = pd.to_numeric(orders_df["unit_price"], errors="coerce")
discount_numeric = pd.to_numeric(orders_df["discount_pct"], errors="coerce")
source_amount_numeric = pd.to_numeric(orders_df["order_amount"], errors="coerce")
expected_amount = (
    quantity_numeric * unit_price_numeric * (1 - discount_numeric / 100)
).round(2)
amount_difference = (source_amount_numeric - expected_amount).abs()
formula_failure_mask = (
    quantity_numeric.isna()
    | unit_price_numeric.isna()
    | discount_numeric.isna()
    | source_amount_numeric.isna()
    | amount_difference.gt(0.01)
)

order_amount_formula_profile_df = pd.DataFrame(
    [
        {
            "check_name": "order amount formula",
            "rows_checked": len(orders_df),
            "rows_failed": int(formula_failure_mask.sum()),
            "failure_percentage": formula_failure_mask.mean() * 100,
        }
    ]
)

formula_samples = orders_df.loc[formula_failure_mask, [
    "order_id",
    "quantity",
    "unit_price",
    "discount_pct",
    "order_amount",
]].copy()
formula_samples["expected_amount"] = expected_amount.loc[formula_failure_mask]
formula_samples["amount_difference"] = amount_difference.loc[formula_failure_mask]
formula_samples["check_name"] = "order amount formula"
rule_sample_rows.append(formula_samples)

order_amount_formula_profile_df

,check_name,rows_checked,rows_failed,failure_percentage
0,order amount formula,2412,18,0.746269


## 2. Delivered orders with blank delivery minutes

This check finds delivered orders whose delivery time is blank after trimming whitespace. It matters because delivery-time metrics cannot be calculated for those rows; it does not decide whether the missing value should be repaired or whether the order should be excluded.

In [20]:
delivery_values = orders_df["delivery_minutes"].astype("string")
delivered_blank_mask = (
    orders_df["order_status"].eq("DELIVERED")
    & delivery_values.str.strip().eq("")
)

delivered_delivery_profile_df = pd.DataFrame(
    [
        {
            "check_name": "DELIVERED orders with blank delivery_minutes",
            "rows_checked": len(orders_df),
            "rows_failed": int(delivered_blank_mask.sum()),
            "failure_percentage": delivered_blank_mask.mean() * 100,
        }
    ]
)

delivered_samples = orders_df.loc[
    delivered_blank_mask,
    ["order_id", "order_status", "delivery_minutes"],
].copy()
delivered_samples["check_name"] = "DELIVERED orders with blank delivery_minutes"
rule_sample_rows.append(delivered_samples)

delivered_delivery_profile_df

,check_name,rows_checked,rows_failed,failure_percentage
0,DELIVERED orders with blank delivery_minutes,2412,8,0.331675


## 3. Allowed order status and payment mode values

These checks compare raw categorical values with the allowed lists from the assignment. They matter because unexpected labels can split reports or prevent reliable grouping; they do not standardise casing, spelling, or whitespace.

In [21]:
allowed_statuses = {"DELIVERED", "CANCELLED", "RETURNED"}
allowed_payment_modes = {"UPI", "Card", "Cash on Delivery", "Wallet"}

status_failure_mask = ~orders_df["order_status"].isin(allowed_statuses)
payment_mode_failure_mask = ~orders_df["payment_mode"].isin(allowed_payment_modes)

allowed_values_profile_df = pd.DataFrame(
    [
        {
            "check_name": "order_status allowed values",
            "rows_checked": len(orders_df),
            "rows_failed": int(status_failure_mask.sum()),
            "failure_percentage": status_failure_mask.mean() * 100,
        },
        {
            "check_name": "payment_mode allowed values",
            "rows_checked": len(orders_df),
            "rows_failed": int(payment_mode_failure_mask.sum()),
            "failure_percentage": payment_mode_failure_mask.mean() * 100,
        },
    ]
)

status_samples = orders_df.loc[
    status_failure_mask,
    ["order_id", "order_status"],
].copy()
status_samples["check_name"] = "order_status allowed values"
rule_sample_rows.append(status_samples)

payment_mode_samples = orders_df.loc[
    payment_mode_failure_mask,
    ["order_id", "payment_mode"],
].copy()
payment_mode_samples["check_name"] = "payment_mode allowed values"
rule_sample_rows.append(payment_mode_samples)

allowed_values_profile_df

,check_name,rows_checked,rows_failed,failure_percentage
0,order_status allowed values,2412,5,0.207297
1,payment_mode allowed values,2412,10,0.414594


## 4. Quantity and discount range profiling

These checks identify quantities at or below zero and discounts outside the stated 0-to-100 range. They matter because both fields feed revenue calculations; the results only identify values for review and do not decide whether they represent returns, corrections, or errors.

In [22]:
quantity_failure_mask = quantity_numeric.le(0)
discount_failure_mask = discount_numeric.lt(0) | discount_numeric.gt(100)

quantity_discount_profile_df = pd.DataFrame(
    [
        {
            "check_name": "quantity <= 0",
            "rows_checked": len(orders_df),
            "rows_failed": int(quantity_failure_mask.sum()),
            "failure_percentage": quantity_failure_mask.mean() * 100,
        },
        {
            "check_name": "discount_pct outside 0-100",
            "rows_checked": len(orders_df),
            "rows_failed": int(discount_failure_mask.sum()),
            "failure_percentage": discount_failure_mask.mean() * 100,
        },
    ]
)

quantity_samples = orders_df.loc[
    quantity_failure_mask,
    ["order_id", "quantity"],
].copy()
quantity_samples["check_name"] = "quantity <= 0"
rule_sample_rows.append(quantity_samples)

discount_samples = orders_df.loc[
    discount_failure_mask,
    ["order_id", "discount_pct"],
].copy()
discount_samples["check_name"] = "discount_pct outside 0-100"
rule_sample_rows.append(discount_samples)

quantity_discount_profile_df

,check_name,rows_checked,rows_failed,failure_percentage
0,quantity <= 0,2412,10,0.414594
1,discount_pct outside 0-100,2412,3,0.124378


## 5. Delivery-time distribution

This section counts delivery times in the requested buckets: `<=60`, `61-120`, and `>120`. Blank or unparsable values are shown separately so they are not silently placed into a numeric bucket; the distribution is descriptive and does not establish a service-level rule.

In [23]:
delivery_numeric = pd.to_numeric(orders_df["delivery_minutes"], errors="coerce")
delivery_bucket = pd.Series(
    "blank_or_invalid",
    index=orders_df.index,
    dtype="string",
)
delivery_bucket.loc[delivery_numeric.le(60)] = "<=60"
delivery_bucket.loc[delivery_numeric.ge(61) & delivery_numeric.le(120)] = "61-120"
delivery_bucket.loc[delivery_numeric.gt(120)] = ">120"

bucket_order = ["<=60", "61-120", ">120", "blank_or_invalid"]
delivery_distribution_df = (
    delivery_bucket.value_counts(dropna=False)
    .reindex(bucket_order, fill_value=0)
    .rename_axis("delivery_bucket")
    .reset_index(name="row_count")
)
delivery_distribution_df["rows_checked"] = len(orders_df)
delivery_distribution_df = delivery_distribution_df[
    ["delivery_bucket", "rows_checked", "row_count"]
]

delivery_over_120_mask = delivery_numeric.gt(120)
delivery_distribution_profile_df = pd.DataFrame(
    [
        {
            "check_name": "delivery_minutes distribution (>120)",
            "rows_checked": len(orders_df),
            "rows_failed": int(delivery_over_120_mask.sum()),
            "failure_percentage": delivery_over_120_mask.mean() * 100,
        }
    ]
)

delivery_samples = orders_df.loc[
    delivery_over_120_mask,
    ["order_id", "delivery_minutes"],
].copy()
delivery_samples["check_name"] = "delivery_minutes distribution (>120)"
rule_sample_rows.append(delivery_samples)

print("Delivery-time bucket counts:")
display(delivery_distribution_df)
print("Delivery-time distribution profile:")
display(delivery_distribution_profile_df)

Delivery-time bucket counts:


,delivery_bucket,rows_checked,row_count
0,<=60,2412,2242
1,61-120,2412,0
2,>120,2412,6
3,blank_or_invalid,2412,164


Delivery-time distribution profile:


,check_name,rows_checked,rows_failed,failure_percentage
0,delivery_minutes distribution (>120),2412,6,0.248756


## 6. Business-rule summary and failing-row samples

This section combines the individual profiling outputs into one summary table. `PASS` means the check found zero failing rows; `INVESTIGATE` means rows were identified for review. The samples are evidence only and do not trigger any data action.

In [24]:
business_rule_profiles = pd.concat(
    [
        order_amount_formula_profile_df,
        delivered_delivery_profile_df,
        allowed_values_profile_df,
        quantity_discount_profile_df,
        delivery_distribution_profile_df,
    ],
    ignore_index=True,
)
business_rule_summary_df = business_rule_profiles[
    ["check_name", "rows_checked", "rows_failed"]
].copy()
business_rule_summary_df["status"] = business_rule_summary_df["rows_failed"].map(
    lambda rows_failed: "PASS" if rows_failed == 0 else "INVESTIGATE"
)

business_rule_samples_df = pd.concat(
    rule_sample_rows,
    ignore_index=True,
    sort=False,
)
business_rule_samples_df = (
    business_rule_samples_df.groupby("check_name", sort=False, group_keys=False)
    .head(5)
    .reset_index(drop=True)
)

print("Business-rule summary:")
display(business_rule_summary_df)

print("Sample failing rows:")
display(business_rule_samples_df)

Business-rule summary:


,check_name,rows_checked,rows_failed,status
0,order amount formula,2412,18,INVESTIGATE
1,DELIVERED orders with blank delivery_minutes,2412,8,INVESTIGATE
2,order_status allowed values,2412,5,INVESTIGATE
3,payment_mode allowed values,2412,10,INVESTIGATE
4,quantity <= 0,2412,10,INVESTIGATE
5,discount_pct outside 0-100,2412,3,INVESTIGATE
6,delivery_minutes distribution (>120),2412,6,INVESTIGATE


Sample failing rows:


,order_id,quantity,unit_price,discount_pct,order_amount,expected_amount,amount_difference,check_name,order_status,delivery_minutes,payment_mode
0,ORD002103,1,492.0,250,492.0,-738.00,1230.00,order amount formula,NaN,NaN,NaN
1,ORD002305,3,265.0,15,6757.5,675.75,6081.75,order amount formula,NaN,NaN,NaN
2,ORD002004,2,336.0,0,6720.0,672.00,6048.00,order amount formula,NaN,NaN,NaN
3,ORD000845,2,248.0,5,4712.0,471.20,4240.80,order amount formula,NaN,NaN,NaN
4,ORD000964,3,279.0,0,8370.0,837.00,7533.00,order amount formula,NaN,NaN,NaN
5,ORD001554,NaN,NaN,NaN,NaN,NaN,NaN,DELIVERED orders with blank delivery_minutes,DELIVERED,,NaN
6,ORD001800,NaN,NaN,NaN,NaN,NaN,NaN,DELIVERED orders with blank delivery_minutes,DELIVERED,,NaN
7,ORD000731,NaN,NaN,NaN,NaN,NaN,NaN,DELIVERED orders with blank delivery_minutes,DELIVERED,,NaN
8,ORD001277,NaN,NaN,NaN,NaN,NaN,NaN,DELIVERED orders with blank delivery_minutes,DELIVERED,,NaN
9,ORD000326,NaN,NaN,NaN,NaN,NaN,NaN,DELIVERED orders with blank delivery_minutes,DELIVERED,,NaN


## Stage 4 observations and questions

## Stage 4 summary table

| check_name | rows_checked | rows_failed | failure_percentage | status |
|---|---:|---:|---:|---|
| order amount formula | 2412 | 18 | 0.75% | INVESTIGATE |
| DELIVERED orders with blank delivery_minutes | 2412 | 8 | 0.33% | INVESTIGATE |
| order_status allowed values | 2412 | 5 | 0.21% | INVESTIGATE |
| payment_mode allowed values | 2412 | 10 | 0.41% | INVESTIGATE |
| quantity <= 0 | 2412 | 10 | 0.41% | INVESTIGATE |
| discount_pct outside 0-100 | 2412 | 3 | 0.12% | INVESTIGATE |
| delivery_minutes distribution (>120) | 2412 | 6 | 0.25% | INVESTIGATE |

## Top 5 most interesting failed checks

1. **Order amount formula:** 18 rows failed. Some source amounts do not match the derived formula, which can affect revenue totals.
2. **Payment mode allowed values:** 10 rows failed. Values such as `Cashh`, `upi`, and `CARD` appear outside the exact allowed list.
3. **Quantity <= 0:** 10 rows failed. The data includes zero and negative quantities, which may represent returns or corrections, but their meaning is not yet confirmed.
4. **DELIVERED orders with blank delivery minutes:** 8 rows failed. These rows cannot contribute a measured delivery time even though their status is `DELIVERED`.
5. **Delivery minutes greater than 120:** 6 rows failed. These values may materially affect delivery-time metrics, but no business threshold has been approved yet.

## My observations

Most Stage 4 failures come from `orders_df` because every cross-column business rule created in this stage uses order-level fields such as quantity, price, discount, status, payment mode, amount, and delivery time.

The other fact and dimension tables were not ignored. They were profiled in earlier stages for structure, missing or blank values, duplicates, candidate keys, numeric/date ranges, and categorical distributions. However, earlier stages did not apply cross-column business rules to payments, customers, products, or stores.

Therefore, the absence of Stage 4 failures for those tables does not prove that they are fully valid. It means those tables were outside the scope of the Stage 4 checks currently implemented.

## Questions

- Which cross-column business rules should be added for payments, customers, products, and stores?
- Should payment checks compare `amount_paid` with the related order amount and payment status?
- Should product checks compare `cost_price` with `mrp`?
- Should customer and store checks compare their attributes with related orders?


# Stage 4.1: Payment Reconciliation Profiling

This stage joins payment records to order amounts using `order_id` and compares successful payment amounts with their order amounts. It is profiling only: no rows are cleaned, modified, quarantined, or deleted.

## Payment reconciliation check

This check evaluates only `SUCCESS` payments and compares `amount_paid` with the joined `order_amount`. The derived order lookup removes only exact duplicate lookup rows to prevent duplicate source rows from multiplying a payment during the join; neither source DataFrame is changed.

In [25]:
order_amount_lookup = orders_df[["order_id", "order_amount"]].drop_duplicates()
payment_reconciliation_df = payments_df.merge(
    order_amount_lookup,
    on="order_id",
    how="left",
    indicator=True,
)

payment_reconciliation_df["amount_paid_numeric"] = pd.to_numeric(
    payment_reconciliation_df["amount_paid"],
    errors="coerce",
)
payment_reconciliation_df["order_amount_numeric"] = pd.to_numeric(
    payment_reconciliation_df["order_amount"],
    errors="coerce",
)
payment_reconciliation_df["difference"] = (
    payment_reconciliation_df["amount_paid_numeric"]
    - payment_reconciliation_df["order_amount_numeric"]
)

success_mask = payment_reconciliation_df["payment_status"].eq("SUCCESS")
order_exists_mask = payment_reconciliation_df["_merge"].eq("both")
amounts_available_mask = (
    payment_reconciliation_df["amount_paid_numeric"].notna()
    & payment_reconciliation_df["order_amount_numeric"].notna()
)
success_failure_mask = success_mask & (
    ~order_exists_mask
    | ~amounts_available_mask
    | payment_reconciliation_df["difference"].abs().gt(0.01)
)
success_matched_mask = success_mask & order_exists_mask & amounts_available_mask

success_payments_checked = int(success_mask.sum())
reconciliation_rows_failed = int(success_failure_mask.sum())
payment_reconciliation_summary = pd.DataFrame(
    [
        {
            "rows_checked": len(payments_df),
            "success_payments_checked": success_payments_checked,
            "rows_failed": reconciliation_rows_failed,
            "failure_percentage": (
                reconciliation_rows_failed / success_payments_checked * 100
                if success_payments_checked
                else 0.0
            ),
            "total_amount_difference": payment_reconciliation_df.loc[
                success_matched_mask,
                "difference",
            ].abs().sum(),
        }
    ]
)

payment_reconciliation_failures_df = payment_reconciliation_df.loc[
    success_failure_mask,
    [
        "payment_id",
        "order_id",
        "payment_status",
        "amount_paid",
        "order_amount",
        "difference",
    ],
].copy()

orphan_payment_rows_df = payment_reconciliation_df.loc[
    ~order_exists_mask,
    payments_df.columns.tolist(),
].copy()
orphan_payment_count = len(orphan_payment_rows_df)
orphan_payment_sample_df = orphan_payment_rows_df.head(5)

print("Payment reconciliation summary:")
display(payment_reconciliation_summary)

print("Sample successful-payment reconciliation failures:")
display(payment_reconciliation_failures_df.head(10))

print("Payment rows whose order_id does not exist in orders_df:")
print("Count:", orphan_payment_count)
display(orphan_payment_sample_df)

Payment reconciliation summary:


,rows_checked,success_payments_checked,rows_failed,failure_percentage,total_amount_difference
0,2228,2116,22,1.039698,785.0


Sample successful-payment reconciliation failures:


,payment_id,order_id,payment_status,amount_paid,order_amount,difference
69,PAY0001568,ORD001031,SUCCESS,-43.05,56.95,-100.0
186,PAY0002219,ORD970544,SUCCESS,730.27,NaN,NaN
304,PAY0000354,ORD001937,SUCCESS,455.0,505.0,-50.0
429,PAY0000779,ORD001840,SUCCESS,2.6,102.6,-100.0
687,PAY0002220,ORD968496,SUCCESS,606.87,NaN,NaN
695,PAY0000323,ORD001864,SUCCESS,182.95,192.95,-10.0
766,PAY0001167,ORD001753,SUCCESS,452.35,502.35,-50.0
770,PAY0001499,ORD001832,SUCCESS,349.3,374.3,-25.0
931,PAY0000909,ORD001360,SUCCESS,103.0,203.0,-100.0
964,PAY0002223,ORD988106,SUCCESS,872.5,NaN,NaN


Payment rows whose order_id does not exist in orders_df:
Count: 8


,payment_id,order_id,payment_ts,payment_mode,amount_paid,payment_status,gateway_ref
186,PAY0002219,ORD970544,2026-09-18 14:22:10,UPI,730.27,SUCCESS,GW1392574404
687,PAY0002220,ORD968496,2026-09-18 14:22:10,UPI,606.87,SUCCESS,GW1503323383
964,PAY0002223,ORD988106,2026-09-18 14:22:10,UPI,872.5,SUCCESS,GW1440914785
1147,PAY0002221,ORD941071,2026-09-18 14:22:10,UPI,618.36,SUCCESS,GW6750407328
1445,PAY0002224,ORD981600,2026-09-18 14:22:10,UPI,536.53,SUCCESS,GW9130004433


# Stage 5: Referential Integrity Profiling

This stage checks whether non-blank foreign-key values in child tables exist in their parent tables. Blank foreign keys are counted separately as completeness issues; no rows are modified, deduplicated, quarantined, or deleted.

## Referential integrity checks

Each check compares a child foreign-key column with the corresponding parent key column. The failure percentage uses only non-blank foreign keys, so missing-key completeness and broken relationships remain visible as separate measures.

In [26]:
def profile_referential_integrity(
    relationship,
    child_dataframe,
    foreign_key_column,
    parent_dataframe,
    parent_key_column,
):
    child_values = child_dataframe[foreign_key_column].astype("string")
    parent_values = parent_dataframe[parent_key_column].astype("string")
    child_blank_mask = (
        child_dataframe[foreign_key_column].isna()
        | child_values.str.strip().eq("")
    )
    parent_non_blank_values = set(
        parent_values.loc[parent_values.str.strip().ne("")].tolist()
    )
    non_blank_mask = ~child_blank_mask
    failure_mask = non_blank_mask & ~child_values.isin(parent_non_blank_values)
    non_blank_keys_checked = int(non_blank_mask.sum())
    rows_failed = int(failure_mask.sum())

    profile_row = {
        "relationship": relationship,
        "rows_checked": len(child_dataframe),
        "blank_foreign_keys": int(child_blank_mask.sum()),
        "non_blank_keys_checked": non_blank_keys_checked,
        "rows_failed": rows_failed,
        "failure_percentage": (
            rows_failed / non_blank_keys_checked * 100
            if non_blank_keys_checked
            else 0.0
        ),
        "status": "PASS" if rows_failed == 0 else "INVESTIGATE",
    }
    sample_rows = child_dataframe.loc[failure_mask].head(10).copy()
    sample_rows["relationship"] = relationship
    return profile_row, sample_rows

referential_relationships = [
    (
        "orders.customer_id -> customers.customer_id",
        orders_df,
        "customer_id",
        customers_df,
        "customer_id",
    ),
    (
        "orders.product_id -> products.product_id",
        orders_df,
        "product_id",
        products_df,
        "product_id",
    ),
    (
        "orders.store_id -> stores.store_id",
        orders_df,
        "store_id",
        stores_df,
        "store_id",
    ),
    (
        "payments.order_id -> orders.order_id",
        payments_df,
        "order_id",
        orders_df,
        "order_id",
    ),
]

referential_profile_rows = []
referential_sample_rows = []
for relationship, child_dataframe, foreign_key_column, parent_dataframe, parent_key_column in referential_relationships:
    profile_row, sample_rows = profile_referential_integrity(
        relationship,
        child_dataframe,
        foreign_key_column,
        parent_dataframe,
        parent_key_column,
    )
    referential_profile_rows.append(profile_row)
    referential_sample_rows.append(sample_rows)

referential_integrity_profile_df = pd.DataFrame(referential_profile_rows)
referential_integrity_summary_df = referential_integrity_profile_df[
    [
        "relationship",
        "rows_checked",
        "blank_foreign_keys",
        "rows_failed",
        "failure_percentage",
        "status",
    ]
].copy()
referential_integrity_samples_df = pd.concat(
    referential_sample_rows,
    ignore_index=True,
    sort=False,
)

print("Referential integrity profile:")
display(referential_integrity_profile_df)

print("Referential integrity summary:")
display(referential_integrity_summary_df)

print("Sample failing rows:")
display(
    pd.concat(
        [
            referential_integrity_samples_df.head(2),
            referential_integrity_samples_df.tail(2),
        ]
    )
)

Referential integrity profile:


,relationship,rows_checked,blank_foreign_keys,non_blank_keys_checked,rows_failed,failure_percentage,status
0,orders.customer_id -> customers.customer_id,2412,18,2394,0,0.000000,PASS
1,orders.product_id -> products.product_id,2412,0,2412,9,0.373134,INVESTIGATE
2,orders.store_id -> stores.store_id,2412,0,2412,5,0.207297,INVESTIGATE
3,payments.order_id -> orders.order_id,2228,0,2228,8,0.359066,INVESTIGATE


Referential integrity summary:


,relationship,rows_checked,blank_foreign_keys,rows_failed,failure_percentage,status
0,orders.customer_id -> customers.customer_id,2412,18,0,0.000000,PASS
1,orders.product_id -> products.product_id,2412,0,9,0.373134,INVESTIGATE
2,orders.store_id -> stores.store_id,2412,0,5,0.207297,INVESTIGATE
3,payments.order_id -> orders.order_id,2228,0,8,0.359066,INVESTIGATE


Sample failing rows:


,order_id,order_ts,customer_id,store_id,product_id,quantity,unit_price,discount_pct,order_amount,payment_mode,order_status,channel,delivery_minutes,relationship,payment_id,payment_ts,amount_paid,payment_status,gateway_ref
0,ORD000986,2026-09-13 19:12:18,C00312,S08,P9455,1,152.0,10,136.8,UPI,DELIVERED,App,9,orders.product_id -> products.product_id,NaN,NaN,NaN,NaN,NaN
1,ORD001058,2026-09-14 18:13:11,C00296,S15,P9212,3,224.0,5,638.4,Wallet,DELIVERED,Web,8,orders.product_id -> products.product_id,NaN,NaN,NaN,NaN,NaN
20,ORD953136,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,UPI,NaN,NaN,NaN,payments.order_id -> orders.order_id,PAY0002222,2026-09-18 14:22:10,581.62,SUCCESS,GW4344391456
21,ORD966342,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,UPI,NaN,NaN,NaN,payments.order_id -> orders.order_id,PAY0002217,2026-09-18 14:22:10,237.84,SUCCESS,GW8488434439


## Stage 5 observations and questions

## Stage 5 profile results

| relationship | rows_checked | blank_foreign_keys | non_blank_keys_checked | rows_failed | failure_percentage | status |
|---|---:|---:|---:|---:|---:|---|
| orders.customer_id -> customers.customer_id | 2412 | 18 | 2394 | 0 | 0.00% | PASS |
| orders.product_id -> products.product_id | 2412 | 0 | 2412 | 9 | 0.37% | INVESTIGATE |
| orders.store_id -> stores.store_id | 2412 | 0 | 2412 | 5 | 0.21% | INVESTIGATE |
| payments.order_id -> orders.order_id | 2228 | 0 | 2228 | 8 | 0.36% | INVESTIGATE |

## Stage 5 summary

| relationship | rows_checked | blank_foreign_keys | rows_failed | failure_percentage | status |
|---|---:|---:|---:|---:|---|
| orders.customer_id -> customers.customer_id | 2412 | 18 | 0 | 0.00% | PASS |
| orders.product_id -> products.product_id | 2412 | 0 | 9 | 0.37% | INVESTIGATE |
| orders.store_id -> stores.store_id | 2412 | 0 | 5 | 0.21% | INVESTIGATE |
| payments.order_id -> orders.order_id | 2228 | 0 | 8 | 0.36% | INVESTIGATE |

Sample failures showed nine order rows with product IDs missing from `products_df`, five order rows with store IDs missing from `stores_df`, and eight payment rows with order IDs missing from `orders_df`. Blank customer foreign keys were reported separately and were not counted as referential failures.

## Question

**Does the payment reconciliation check fall under Stage 5?**

Not exactly. The `payments.order_id -> orders.order_id` existence check belongs to Stage 5 because it is referential integrity profiling. The payment reconciliation check from Stage 4.1 is separate: it compares successful `amount_paid` values with the related `order_amount`. Both use payments and orders, but they answer different questions:

- Stage 5 asks: **Does the referenced order exist?**
- Stage 4.1 asks: **Does the successful payment amount match the order amount?**


# Stage 6: Part 0 Final Deliverables

This stage reuses the profiling results from Stages 1-5 to assemble the three Part 0 deliverables: the Data Issues Log, the Candidate Validation Rules, and the three best client questions. No source DataFrame is modified, cleaned, standardised, deduplicated, quarantined, or deleted in this stage. A small number of checks that were not yet run anywhere above (product pricing, store capacity, the remaining duplicate-key groups, and the remaining date columns) are executed here only to extract exact affected row keys with evidence; they follow the same non-destructive profiling style as the earlier stages.

## Deliverable 1: Data Issues Log

Every confirmed, evidence-backed problem is recorded as one row per affected source row per failed check in `data_issues_log_df`. The `status` column marks every row as `"Open"`: an issue has been logged, but Part 0 does not quarantine, correct, or drop any source row. `severity` follows the assignment's definitions (`Critical` = should be quarantined before it reaches joins/revenue/finance reporting, `Warning` = may continue to staging after logging, `Pending Client Decision` = correct treatment depends on an unconfirmed business definition). The helper function below only assembles dictionaries from values already computed by Stages 1-5 (or from the small number of additional checks documented in this stage); it does not alter any source DataFrame. All exported files in this stage are written to a dedicated `part0_outputs/` subfolder next to the source CSVs, so re-running this notebook never adds an extra `.csv` file to `dataset_dir` and never breaks the Stage 1 "exactly six files" assertion.

In [43]:
def make_issue_row(file, row_key, column, problem, observed_value, quality_dimension, proposed_rule, severity, evidence_check):
    """Build one Data Issues Log row. issue_id is assigned later, after sorting."""
    return {
        "file": file,
        "row_key": row_key,
        "column": column,
        "problem": problem,
        "observed_value": observed_value,
        "quality_dimension": quality_dimension,
        "proposed_rule": proposed_rule,
        "severity": severity,
        "evidence_check": evidence_check,
        "status": "Open",
    }


issue_rows = []

# --- orders.csv -------------------------------------------------------

# 1. Blank order_ts (Completeness)
order_ts_blank_mask = orders_df["order_ts"].astype("string").str.strip().eq("")
for _, row in orders_df.loc[order_ts_blank_mask].iterrows():
    issue_rows.append(make_issue_row(
        "orders.csv", row["order_id"], "order_ts",
        "order_ts is blank; the order cannot be placed in time for freshness or daily reporting.",
        "order_ts=''",
        "Completeness", "order_ts must be present", "Critical",
        "Stage 2 missing-value profiling",
    ))

# 2. Blank customer_id (Completeness; treatment depends on Client Question 2)
customer_id_blank_mask = orders_df["customer_id"].astype("string").str.strip().eq("")
for _, row in orders_df.loc[customer_id_blank_mask].iterrows():
    issue_rows.append(make_issue_row(
        "orders.csv", row["order_id"], "customer_id",
        "customer_id is blank; the order cannot be attributed to a known customer for customer-level reporting.",
        "customer_id=''",
        "Completeness", "customer_id must be present", "Pending Client Decision",
        "Stage 2 missing-value profiling",
    ))

# 3. DELIVERED order with blank delivery_minutes (Completeness)
for _, row in orders_df.loc[delivered_blank_mask].iterrows():
    issue_rows.append(make_issue_row(
        "orders.csv", row["order_id"], "delivery_minutes",
        "order_status is DELIVERED but delivery_minutes is blank, so delivery-time metrics cannot be computed for this order.",
        f"order_status={row['order_status']}, delivery_minutes=''",
        "Completeness", "DELIVERED orders must have delivery_minutes", "Critical",
        "Stage 4 delivered-orders blank delivery_minutes check",
    ))

# 4. Duplicate order_id / exact duplicate order rows (Uniqueness) - every row in a repeated-key group, including the first occurrence
for order_id, group in orders_exact_key_duplicate_rows.groupby("order_id", sort=False):
    for _, row in group.iterrows():
        issue_rows.append(make_issue_row(
            "orders.csv", row["order_id"], "order_id",
            "order_id is repeated across multiple exact-duplicate full rows.",
            f"order_id {order_id} repeated {len(group)} times (identical full rows)",
            "Uniqueness", "order_id must be unique", "Critical",
            "Stage 2 candidate-key profiling",
        ))

# 5 & 6. Invalid / future order_ts (Validity / Timeliness)
order_ts_values = orders_df["order_ts"].astype("string")
order_ts_non_blank_mask = order_ts_values.notna() & order_ts_values.str.strip().ne("")
order_ts_parsed = pd.to_datetime(order_ts_values, errors="coerce", format="mixed")
order_ts_invalid_mask = order_ts_non_blank_mask & order_ts_parsed.isna()
order_ts_future_mask = order_ts_parsed.notna() & (order_ts_parsed > future_reference_date)

for _, row in orders_df.loc[order_ts_invalid_mask].iterrows():
    issue_rows.append(make_issue_row(
        "orders.csv", row["order_id"], "order_ts",
        "order_ts does not parse as a valid timestamp.",
        f"order_ts={row['order_ts']!r}",
        "Validity", "order_ts must parse as a valid timestamp", "Critical",
        "Stage 3 date profiling",
    ))

for _, row in orders_df.loc[order_ts_future_mask].iterrows():
    issue_rows.append(make_issue_row(
        "orders.csv", row["order_id"], "order_ts",
        f"order_ts is later than the fixed profiling reference date ({future_reference_date.date()}).",
        f"order_ts={row['order_ts']}",
        "Timeliness", "order_ts must not be future-dated", "Critical",
        "Stage 3 date profiling",
    ))

# 7. Invalid order_status values (Validity)
for _, row in orders_df.loc[status_failure_mask].iterrows():
    issue_rows.append(make_issue_row(
        "orders.csv", row["order_id"], "order_status",
        f"order_status is not one of the approved values {sorted(allowed_statuses)}.",
        f"order_status={row['order_status']!r}",
        "Validity", "order_status must be in the approved list", "Critical",
        "Stage 4 allowed order_status values check",
    ))

# 8. Invalid payment_mode values (Validity)
for _, row in orders_df.loc[payment_mode_failure_mask].iterrows():
    issue_rows.append(make_issue_row(
        "orders.csv", row["order_id"], "payment_mode",
        f"payment_mode is not one of the approved values {sorted(allowed_payment_modes)}.",
        f"payment_mode={row['payment_mode']!r}",
        "Validity", "payment_mode must be in the approved list", "Warning",
        "Stage 4 allowed payment_mode values check",
    ))

# 9. quantity <= 0 (Accuracy; treatment depends on Client Question 1)
for _, row in orders_df.loc[quantity_failure_mask].iterrows():
    issue_rows.append(make_issue_row(
        "orders.csv", row["order_id"], "quantity",
        "quantity is zero or negative.",
        f"quantity={row['quantity']}",
        "Accuracy", "quantity must be greater than zero", "Pending Client Decision",
        "Stage 4 quantity <= 0 check",
    ))

# 10. discount_pct outside 0-100 (Validity)
for _, row in orders_df.loc[discount_failure_mask].iterrows():
    issue_rows.append(make_issue_row(
        "orders.csv", row["order_id"], "discount_pct",
        "discount_pct is outside the valid 0-100 range.",
        f"discount_pct={row['discount_pct']}",
        "Validity", "discount_pct must be between 0 and 100", "Critical",
        "Stage 4 discount_pct range check",
    ))

# 11. order_amount formula mismatch (Accuracy) - comma-separated column list per the assignment's example
for _, row in orders_df.loc[formula_failure_mask].iterrows():
    index = row.name
    issue_rows.append(make_issue_row(
        "orders.csv", row["order_id"], "quantity,unit_price,discount_pct,order_amount",
        "order_amount does not match quantity * unit_price * (1 - discount_pct / 100), rounded to 2 decimals.",
        (
            f"quantity={row['quantity']}, unit_price={row['unit_price']}, "
            f"discount_pct={row['discount_pct']}, order_amount={row['order_amount']}, "
            f"expected_amount={expected_amount.loc[index]}, "
            f"abs_difference={amount_difference.loc[index]}"
        ),
        "Accuracy", "order_amount must match the business formula", "Critical",
        "Stage 4 order amount formula check",
    ))

# 12. delivery_minutes > 120 (Accuracy; treated as an SLA warning pending Client Question 3)
for _, row in orders_df.loc[delivery_over_120_mask].iterrows():
    issue_rows.append(make_issue_row(
        "orders.csv", row["order_id"], "delivery_minutes",
        "delivery_minutes exceeds 120 minutes; no approved SLA threshold exists yet.",
        f"delivery_minutes={row['delivery_minutes']}",
        "Accuracy", "delivery_minutes must not exceed the approved threshold", "Warning",
        "Stage 4 delivery_minutes distribution check",
    ))

# 13 & 14. product_id / store_id not found in the parent file (Consistency) - recomputed at full fidelity
# because referential_integrity_samples_df in Stage 5 is capped at head(10) per relationship.
product_id_values = orders_df["product_id"].astype("string")
product_id_non_blank_mask = product_id_values.notna() & product_id_values.str.strip().ne("")
known_product_ids = set(products_df["product_id"].astype("string").str.strip().loc[lambda s: s.ne("")])
product_fk_fail_mask = product_id_non_blank_mask & ~product_id_values.isin(known_product_ids)

store_id_values = orders_df["store_id"].astype("string")
store_id_non_blank_mask = store_id_values.notna() & store_id_values.str.strip().ne("")
known_store_ids = set(stores_df["store_id"].astype("string").str.strip().loc[lambda s: s.ne("")])
store_fk_fail_mask = store_id_non_blank_mask & ~store_id_values.isin(known_store_ids)

for _, row in orders_df.loc[product_fk_fail_mask].iterrows():
    issue_rows.append(make_issue_row(
        "orders.csv", row["order_id"], "product_id",
        "product_id does not exist in products.csv.",
        f"product_id={row['product_id']!r}",
        "Consistency", "product_id must exist in products.csv", "Critical",
        "Stage 5 referential integrity (orders.product_id -> products.product_id)",
    ))

for _, row in orders_df.loc[store_fk_fail_mask].iterrows():
    issue_rows.append(make_issue_row(
        "orders.csv", row["order_id"], "store_id",
        "store_id does not exist in stores.csv.",
        f"store_id={row['store_id']!r}",
        "Consistency", "store_id must exist in stores.csv", "Critical",
        "Stage 5 referential integrity (orders.store_id -> stores.store_id)",
    ))

print(f"orders.csv issue rows logged so far: {len(issue_rows)}")

orders.csv issue rows logged so far: 129


In [44]:
# --- payments.csv -------------------------------------------------------

# 1. Duplicate payment_id / exact duplicate payment rows (Uniqueness)
for payment_id, group in payments_exact_key_duplicate_rows.groupby("payment_id", sort=False):
    for _, row in group.iterrows():
        issue_rows.append(make_issue_row(
            "payments.csv", row["payment_id"], "payment_id",
            "payment_id is repeated across multiple exact-duplicate full rows.",
            f"payment_id {payment_id} repeated {len(group)} times (identical full rows)",
            "Uniqueness", "payment_id must be unique", "Critical",
            "Stage 2 candidate-key profiling",
        ))

# 2. Future payment_ts (Timeliness) - recalculated at row level; Stage 3 only stored the count
payment_ts_parsed = pd.to_datetime(payments_df["payment_ts"].astype("string"), errors="coerce", format="mixed")
payment_ts_future_mask = payment_ts_parsed.notna() & (payment_ts_parsed > future_reference_date)
for _, row in payments_df.loc[payment_ts_future_mask].iterrows():
    issue_rows.append(make_issue_row(
        "payments.csv", row["payment_id"], "payment_ts",
        f"payment_ts is later than the fixed profiling reference date ({future_reference_date.date()}).",
        f"payment_ts={row['payment_ts']}",
        "Timeliness", "payment_ts must not be future-dated", "Critical",
        "Stage 3 date profiling",
    ))

# 3. Payment order_id not found in orders.csv (Consistency) - orphan payments only
for _, row in orphan_payment_rows_df.iterrows():
    issue_rows.append(make_issue_row(
        "payments.csv", row["payment_id"], "order_id",
        "payment order_id does not exist in orders.csv; there is no order_amount to reconcile against.",
        f"order_id={row['order_id']!r}, payment_status={row['payment_status']}, amount_paid={row['amount_paid']}",
        "Consistency", "payment order_id must exist in orders.csv", "Critical",
        "Stage 4.1 payment reconciliation (orphan payments)",
    ))

# 4. SUCCESS payment amount mismatch (Consistency) - matched (non-orphan) orders only, so orphans are
# never double-counted as amount mismatches, per the assignment's reconciliation requirement.
amount_mismatch_mask = (
    success_mask
    & order_exists_mask
    & amounts_available_mask
    & payment_reconciliation_df["difference"].abs().gt(0.01)
)
for _, row in payment_reconciliation_df.loc[amount_mismatch_mask].iterrows():
    issue_rows.append(make_issue_row(
        "payments.csv", row["payment_id"], "amount_paid",
        "payment_status is SUCCESS but amount_paid does not equal the related order_amount (tolerance 0.01).",
        (
            f"order_id={row['order_id']}, amount_paid={row['amount_paid']}, "
            f"order_amount={row['order_amount']}, signed_difference={row['difference']}"
        ),
        "Consistency", "SUCCESS payment amount must equal order_amount", "Critical",
        "Stage 4.1 payment reconciliation (SUCCESS amount mismatch)",
    ))

print(f"issue rows logged so far (orders.csv + payments.csv): {len(issue_rows)}")

# --- Investigation note: gateway_ref is NOT logged as a Data Issues Log row ---
gateway_ref_blank_mask = payments_df["gateway_ref"].astype("string").str.strip().eq("")
gateway_ref_blank_modes = payments_df.loc[gateway_ref_blank_mask, "payment_mode"].value_counts()
print(
    "Investigation note: gateway_ref is blank for",
    int(gateway_ref_blank_mask.sum()),
    "payments.csv rows, and every one of them has payment_mode:",
    gateway_ref_blank_modes.to_dict(),
    "\nBecause Cash on Delivery payments have no payment gateway, this blank pattern is fully explained"
    " by payment_mode and is not logged as a Completeness issue in data_issues_log_df.",
)

issue rows logged so far (orders.csv + payments.csv): 166
Investigation note: gateway_ref is blank for 301 payments.csv rows, and every one of them has payment_mode: {'Cash on Delivery': 301} 
Because Cash on Delivery payments have no payment gateway, this blank pattern is fully explained by payment_mode and is not logged as a Completeness issue in data_issues_log_df.


In [45]:
# --- customers.csv -------------------------------------------------------

# 1. Blank city (Completeness)
city_blank_mask = customers_df["city"].astype("string").str.strip().eq("")
for _, row in customers_df.loc[city_blank_mask].iterrows():
    issue_rows.append(make_issue_row(
        "customers.csv", row["customer_id"], "city",
        "city is blank; city-level reporting cannot include this customer.",
        "city=''",
        "Completeness", "city should not be blank", "Warning",
        "Stage 2 missing-value profiling",
    ))

# 2. Duplicate customer_id (Uniqueness) - every row in a repeated-key group, including the first occurrence
customers_exact_key_duplicate_rows, customers_non_identical_key_rows = split_repeated_key_groups(
    customers_df, "customer_id",
)
for customer_id, group in customers_non_identical_key_rows.groupby("customer_id", sort=False):
    for _, row in group.iterrows():
        issue_rows.append(make_issue_row(
            "customers.csv", row["customer_id"], "customer_id",
            "customer_id is repeated with differing phone, area, or pincode values across the repeated rows.",
            f"customer_id {customer_id} repeated {len(group)} times (non-identical rows)",
            "Uniqueness", "customer_id must be unique in customers.csv", "Critical",
            "Stage 2 candidate-key profiling",
        ))
for customer_id, group in customers_exact_key_duplicate_rows.groupby("customer_id", sort=False):
    for _, row in group.iterrows():
        issue_rows.append(make_issue_row(
            "customers.csv", row["customer_id"], "customer_id",
            "customer_id is repeated across multiple exact-duplicate full rows.",
            f"customer_id {customer_id} repeated {len(group)} times (identical full rows)",
            "Uniqueness", "customer_id must be unique in customers.csv", "Critical",
            "Stage 2 candidate-key profiling",
        ))

# 3. Future signup_date (Timeliness)
signup_date_parsed = pd.to_datetime(customers_df["signup_date"].astype("string"), errors="coerce", format="mixed")
signup_date_future_mask = signup_date_parsed.notna() & (signup_date_parsed > future_reference_date)
for _, row in customers_df.loc[signup_date_future_mask].iterrows():
    issue_rows.append(make_issue_row(
        "customers.csv", row["customer_id"], "signup_date",
        f"signup_date is later than the fixed profiling reference date ({future_reference_date.date()}).",
        f"signup_date={row['signup_date']}",
        "Timeliness", "signup_date must not be future-dated", "Warning",
        "Stage 3 date profiling",
    ))

# 4. loyalty_tier casing variants (Validity) - evidence already exists in the Stage 3 categorical profile
allowed_loyalty_tiers = {"Bronze", "Silver", "Gold", "Platinum"}
loyalty_tier_failure_mask = ~customers_df["loyalty_tier"].isin(allowed_loyalty_tiers)
for _, row in customers_df.loc[loyalty_tier_failure_mask].iterrows():
    issue_rows.append(make_issue_row(
        "customers.csv", row["customer_id"], "loyalty_tier",
        "loyalty_tier is a casing variant of an approved tier name rather than one of the exact approved values.",
        f"loyalty_tier={row['loyalty_tier']!r}",
        "Validity", "loyalty_tier must be in the approved list", "Warning",
        "Stage 3 categorical profiling",
    ))

print(f"issue rows logged so far (+ customers.csv): {len(issue_rows)}")

issue rows logged so far (+ customers.csv): 193


In [46]:
# --- products.csv -------------------------------------------------------

# 1. Blank category (Completeness)
category_blank_mask = products_df["category"].astype("string").str.strip().eq("")
for _, row in products_df.loc[category_blank_mask].iterrows():
    issue_rows.append(make_issue_row(
        "products.csv", row["product_id"], "category",
        "category is blank; category-level reporting cannot include this product.",
        "category=''",
        "Completeness", "category should not be blank", "Warning",
        "Stage 2 missing-value profiling",
    ))

# 2. Duplicate product_id (Uniqueness) - every row in a repeated-key group, including the first occurrence
products_exact_key_duplicate_rows, products_non_identical_key_rows = split_repeated_key_groups(
    products_df, "product_id",
)
for product_id, group in products_non_identical_key_rows.groupby("product_id", sort=False):
    for _, row in group.iterrows():
        issue_rows.append(make_issue_row(
            "products.csv", row["product_id"], "product_id",
            "product_id is repeated with a differing product_name/mrp across the repeated rows (an '(old)' variant reuses the same product_id).",
            f"product_id {product_id} repeated {len(group)} times (non-identical rows)",
            "Uniqueness", "product_id must be unique in products.csv", "Critical",
            "Stage 2 candidate-key profiling",
        ))
for product_id, group in products_exact_key_duplicate_rows.groupby("product_id", sort=False):
    for _, row in group.iterrows():
        issue_rows.append(make_issue_row(
            "products.csv", row["product_id"], "product_id",
            "product_id is repeated across multiple exact-duplicate full rows.",
            f"product_id {product_id} repeated {len(group)} times (identical full rows)",
            "Uniqueness", "product_id must be unique in products.csv", "Critical",
            "Stage 2 candidate-key profiling",
        ))

# 3. mrp <= 0 and cost_price > mrp (Accuracy) - run here for the first time, only because profiling
# evidence (below) shows actual failures; no issue is added if a mask finds nothing.
mrp_numeric = pd.to_numeric(products_df["mrp"], errors="coerce")
cost_price_numeric = pd.to_numeric(products_df["cost_price"], errors="coerce")
mrp_non_positive_mask = mrp_numeric.notna() & mrp_numeric.le(0)
cost_price_above_mrp_mask = mrp_numeric.notna() & cost_price_numeric.notna() & cost_price_numeric.gt(mrp_numeric)

for _, row in products_df.loc[mrp_non_positive_mask].iterrows():
    issue_rows.append(make_issue_row(
        "products.csv", row["product_id"], "mrp",
        "mrp is zero or negative, which is not a valid retail price.",
        f"mrp={row['mrp']}",
        "Accuracy", "mrp must be greater than zero", "Critical",
        "Stage 6 product pricing check",
    ))

for _, row in products_df.loc[cost_price_above_mrp_mask].iterrows():
    issue_rows.append(make_issue_row(
        "products.csv", row["product_id"], "cost_price,mrp",
        "cost_price is greater than mrp, implying a negative gross margin on every sale of this product.",
        f"cost_price={row['cost_price']}, mrp={row['mrp']}",
        "Accuracy", "cost_price should not exceed mrp", "Warning",
        "Stage 6 product pricing check",
    ))

print(f"issue rows logged so far (+ products.csv): {len(issue_rows)}")

issue rows logged so far (+ products.csv): 206


In [47]:
# --- stores.csv -------------------------------------------------------

# 1. Blank zone (Completeness)
zone_blank_mask = stores_df["zone"].astype("string").str.strip().eq("")
for _, row in stores_df.loc[zone_blank_mask].iterrows():
    issue_rows.append(make_issue_row(
        "stores.csv", row["store_id"], "zone",
        "zone is blank; zone-level reporting cannot include this store.",
        "zone=''",
        "Completeness", "zone should not be blank", "Warning",
        "Stage 2 missing-value profiling",
    ))

# 2. Duplicate store_id - the Stage 2 candidate-key profile found none, so no issue is added here.
stores_exact_key_duplicate_rows, stores_non_identical_key_rows = split_repeated_key_groups(stores_df, "store_id")
assert stores_exact_key_duplicate_rows.empty and stores_non_identical_key_rows.empty, (
    "Expected no duplicate store_id values based on Stage 2 candidate-key profiling."
)

# 3. capacity_orders_per_day <= 0 (Accuracy) - run here for the first time, only because profiling
# evidence (below) shows an actual failure.
capacity_numeric = pd.to_numeric(stores_df["capacity_orders_per_day"], errors="coerce")
capacity_non_positive_mask = capacity_numeric.notna() & capacity_numeric.le(0)
for _, row in stores_df.loc[capacity_non_positive_mask].iterrows():
    issue_rows.append(make_issue_row(
        "stores.csv", row["store_id"], "capacity_orders_per_day",
        "capacity_orders_per_day is zero or negative, which is not a valid daily order capacity.",
        f"capacity_orders_per_day={row['capacity_orders_per_day']}",
        "Accuracy", "capacity_orders_per_day must be greater than zero", "Warning",
        "Stage 6 store capacity check",
    ))

print(f"issue rows logged so far (+ stores.csv): {len(issue_rows)}")

issue rows logged so far (+ stores.csv): 208


In [48]:
# --- orders_2026-10-01_increment.csv -------------------------------------------------------
# No new checks are run here. This cell only re-reads the already-computed Stage 1-3 results for
# orders_increment_df to confirm, with evidence, that no structural/key issue exists in the completed
# checks. No row is added to data_issues_log_df for this file.

increment_duplicate_occurrences = int(
    duplicate_profile_df.loc[duplicate_profile_df["dataset"] == "orders_increment_df", "duplicate_occurrences"].iloc[0]
)
increment_key_profile = candidate_key_profile_df.loc[candidate_key_profile_df["dataset"] == "orders_increment_df"].iloc[0]
increment_numeric_invalid_total = int(
    numeric_profile_df.loc[numeric_profile_df["dataset"] == "orders_increment_df", "invalid_numeric_count"].sum()
)
increment_date_profile = date_profile_df.loc[date_profile_df["dataset"] == "orders_increment_df"].iloc[0]

increment_checks_clean = (
    schema_matches_exactly
    and increment_duplicate_occurrences == 0
    and int(increment_key_profile["missing_key_count"]) == 0
    and int(increment_key_profile["duplicate_occurrences_after_first"]) == 0
    and increment_numeric_invalid_total == 0
    and int(increment_date_profile["invalid_date_count"]) == 0
    and int(increment_date_profile["future_date_count"]) == 0
)

print("Profiling note: orders_2026-10-01_increment.csv")
print("-" * 60)
print(f"- Schema matches orders.csv exactly: {schema_matches_exactly}")
print(f"- Exact full-row duplicates: {increment_duplicate_occurrences}")
print(f"- Missing/duplicate order_id (candidate key): "
      f"{int(increment_key_profile['missing_key_count'])} missing, "
      f"{int(increment_key_profile['duplicate_occurrences_after_first'])} duplicate")
print(f"- Invalid numeric values across all numeric columns: {increment_numeric_invalid_total}")
print(f"- Invalid order_ts values: {int(increment_date_profile['invalid_date_count'])}")
print(f"- Future-dated order_ts values: {int(increment_date_profile['future_date_count'])}")
print()
if increment_checks_clean:
    print(
        "No structural/key issue was found in orders_2026-10-01_increment.csv in the completed checks. "
        "Business-rule checks (Stage 4) and referential-integrity checks (Stage 5) were not run against "
        "this file in this notebook, so this statement covers structural and key checks only. "
        "No row from this file is added to data_issues_log_df."
    )
else:
    print(
        "WARNING: one or more structural/key counts above is non-zero; re-examine before assuming "
        "the increment file has no structural issues."
    )

Profiling note: orders_2026-10-01_increment.csv
------------------------------------------------------------
- Schema matches orders.csv exactly: True
- Exact full-row duplicates: 0
- Missing/duplicate order_id (candidate key): 0 missing, 0 duplicate
- Invalid numeric values across all numeric columns: 0
- Invalid order_ts values: 0
- Future-dated order_ts values: 0

No structural/key issue was found in orders_2026-10-01_increment.csv in the completed checks. Business-rule checks (Stage 4) and referential-integrity checks (Stage 5) were not run against this file in this notebook, so this statement covers structural and key checks only. No row from this file is added to data_issues_log_df.


In [49]:
data_issues_log_df = pd.DataFrame(issue_rows)

data_issues_log_df = data_issues_log_df.sort_values(
    by=["file", "row_key", "quality_dimension", "column"],
    kind="stable",
).reset_index(drop=True)

data_issues_log_df.insert(
    0, "issue_id", [f"ISSUE-{index + 1:04d}" for index in range(len(data_issues_log_df))],
)

data_issues_log_df = data_issues_log_df[
    [
        "issue_id", "file", "row_key", "column", "problem", "observed_value",
        "quality_dimension", "proposed_rule", "severity", "evidence_check", "status",
    ]
]

allowed_quality_dimensions = {
    "Completeness", "Uniqueness", "Validity", "Accuracy", "Consistency", "Timeliness",
}
allowed_severities = {"Critical", "Warning", "Pending Client Decision"}

file_to_source = {
    "orders.csv": (orders_df, "order_id"),
    "payments.csv": (payments_df, "payment_id"),
    "customers.csv": (customers_df, "customer_id"),
    "products.csv": (products_df, "product_id"),
    "stores.csv": (stores_df, "store_id"),
}

assert data_issues_log_df["issue_id"].is_unique, "issue_id must be unique."
assert data_issues_log_df["file"].astype("string").str.strip().ne("").all(), "file must be populated."
assert data_issues_log_df["row_key"].astype("string").str.strip().ne("").all(), "row_key must be populated."
assert set(data_issues_log_df["quality_dimension"]).issubset(allowed_quality_dimensions), (
    "quality_dimension must only use the approved values."
)
assert set(data_issues_log_df["severity"]).issubset(allowed_severities), (
    "severity must only use the approved values."
)
assert data_issues_log_df["proposed_rule"].astype("string").str.strip().ne("").all(), (
    "proposed_rule must be populated."
)
assert data_issues_log_df["evidence_check"].astype("string").str.strip().ne("").all(), (
    "every issue must cite the evidence check that produced it."
)

for file_name, (source_dataframe, key_column) in file_to_source.items():
    known_keys = set(source_dataframe[key_column].astype("string"))
    file_rows = data_issues_log_df.loc[data_issues_log_df["file"] == file_name]
    unknown_keys = set(file_rows["row_key"]) - known_keys
    assert not unknown_keys, f"{file_name}: row_key values not found in the source DataFrame: {unknown_keys}"

issues_by_file = (
    data_issues_log_df.groupby("file", sort=False).size().rename("issue_instances").reset_index()
)
issues_by_quality_dimension = (
    data_issues_log_df.groupby("quality_dimension", sort=False).size().rename("issue_instances").reset_index()
)
issues_by_severity = (
    data_issues_log_df.groupby("severity", sort=False).size().rename("issue_instances").reset_index()
)
issues_by_evidence_check = (
    data_issues_log_df.groupby("evidence_check", sort=False).size().rename("issue_instances")
    .reset_index().sort_values("issue_instances", ascending=False).reset_index(drop=True)
)
distinct_affected_rows_by_file = (
    data_issues_log_df.groupby("file", sort=False)["row_key"].nunique().rename("distinct_affected_rows").reset_index()
)

main_source_files = {"orders.csv", "payments.csv", "customers.csv", "products.csv", "stores.csv"}
files_with_issues = set(data_issues_log_df["file"].unique())
files_without_issues = main_source_files - files_with_issues
if files_without_issues:
    print(f"NOTE: the following main source files have no evidence-backed issue: {sorted(files_without_issues)}")
else:
    print("All five main source files have at least one evidence-backed issue.")

total_issue_instances = len(data_issues_log_df)
total_distinct_affected_rows = int(distinct_affected_rows_by_file["distinct_affected_rows"].sum())

with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.width", None):
    print("Complete Data Issues Log:")
    display(data_issues_log_df)

print("Issues by file:")
display(issues_by_file)

print("Issues by quality dimension:")
display(issues_by_quality_dimension)

print("Issues by severity:")
display(issues_by_severity)

print("Issues by evidence check:")
display(issues_by_evidence_check)

print("Distinct affected source rows by file:")
display(distinct_affected_rows_by_file)

print(f"Total issue instances: {total_issue_instances}")
print(f"Total distinct affected source rows: {total_distinct_affected_rows}")

part0_outputs_dir = dataset_dir / "part0_outputs"
part0_outputs_dir.mkdir(parents=True, exist_ok=True)

data_issues_log_df.to_csv(part0_outputs_dir / "part0_data_issues_log.csv", index=False)
print(f"Exported: {(part0_outputs_dir / 'part0_data_issues_log.csv').resolve()}")

All five main source files have at least one evidence-backed issue.
Complete Data Issues Log:


,issue_id,file,row_key,column,problem,observed_value,quality_dimension,proposed_rule,severity,evidence_check,status
0,ISSUE-0001,customers.csv,C00031,loyalty_tier,loyalty_tier is a casing variant of an approve...,loyalty_tier='bronze',Validity,loyalty_tier must be in the approved list,Warning,Stage 3 categorical profiling,Open
1,ISSUE-0002,customers.csv,C00032,loyalty_tier,loyalty_tier is a casing variant of an approve...,loyalty_tier='bronze',Validity,loyalty_tier must be in the approved list,Warning,Stage 3 categorical profiling,Open
2,ISSUE-0003,customers.csv,C00036,customer_id,"customer_id is repeated with differing phone, ...",customer_id C00036 repeated 2 times (non-ident...,Uniqueness,customer_id must be unique in customers.csv,Critical,Stage 2 candidate-key profiling,Open
3,ISSUE-0004,customers.csv,C00036,customer_id,"customer_id is repeated with differing phone, ...",customer_id C00036 repeated 2 times (non-ident...,Uniqueness,customer_id must be unique in customers.csv,Critical,Stage 2 candidate-key profiling,Open
4,ISSUE-0005,customers.csv,C00038,signup_date,signup_date is later than the fixed profiling ...,signup_date=2026-12-20,Timeliness,signup_date must not be future-dated,Warning,Stage 3 date profiling,Open
5,ISSUE-0006,customers.csv,C00092,loyalty_tier,loyalty_tier is a casing variant of an approve...,loyalty_tier='BRONZE',Validity,loyalty_tier must be in the approved list,Warning,Stage 3 categorical profiling,Open
6,ISSUE-0007,customers.csv,C00103,customer_id,"customer_id is repeated with differing phone, ...",customer_id C00103 repeated 2 times (non-ident...,Uniqueness,customer_id must be unique in customers.csv,Critical,Stage 2 candidate-key profiling,Open
7,ISSUE-0008,customers.csv,C00103,customer_id,"customer_id is repeated with differing phone, ...",customer_id C00103 repeated 2 times (non-ident...,Uniqueness,customer_id must be unique in customers.csv,Critical,Stage 2 candidate-key profiling,Open
8,ISSUE-0009,customers.csv,C00133,loyalty_tier,loyalty_tier is a casing variant of an approve...,loyalty_tier='BRONZE',Validity,loyalty_tier must be in the approved list,Warning,Stage 3 categorical profiling,Open
9,ISSUE-0010,customers.csv,C00136,loyalty_tier,loyalty_tier is a casing variant of an approve...,loyalty_tier='BRONZE',Validity,loyalty_tier must be in the approved list,Warning,Stage 3 categorical profiling,Open


Issues by file:


,file,issue_instances
0,customers.csv,27
1,orders.csv,129
2,payments.csv,37
3,products.csv,13
4,stores.csv,2


Issues by quality dimension:


,quality_dimension,issue_instances
0,Validity,31
1,Uniqueness,46
2,Timeliness,17
3,Completeness,37
4,Accuracy,41
5,Consistency,36


... (3 outputs left out)

,evidence_check,issue_instances
0,Stage 2 candidate-key profiling,46
1,Stage 2 missing-value profiling,29
2,Stage 3 date profiling,22
3,Stage 4 order amount formula check,18
4,Stage 4.1 payment reconciliation (SUCCESS amou...,14
5,Stage 4 allowed payment_mode values check,10
6,Stage 4 quantity <= 0 check,10
7,Stage 5 referential integrity (orders.product_...,9
8,Stage 4.1 payment reconciliation (orphan payme...,8
9,Stage 3 categorical profiling,8


Distinct affected source rows by file:


,file,distinct_affected_rows
0,customers.csv,22
1,orders.csv,114
2,payments.csv,33
3,products.csv,10
4,stores.csv,2


Total issue instances: 208
Total distinct affected source rows: 181
Exported: /Users/irfan_shaik@optum.com/Downloads/IIT Roorkee/Assignment_2/IIT_Roorkie_CEC_Futurenece_Data_Pipeline_to_ML_Model/assignment/nammamart_dataset/part0_outputs/part0_data_issues_log.csv


## Deliverable 2: Candidate Validation Rules

`candidate_validation_rules_df` holds one row per distinct validation concept (not one row per failing record). Each rule's `evidence_rows_failed` count is pulled directly from `data_issues_log_df` by matching on `rule_name` (which was written identically to `proposed_rule` while building the log), or from the Stage 1-5 profiling results for rules that currently have zero failures (safeguard rules and the gateway_ref/Cash-on-Delivery pattern). `failure_action` follows the assignment's safe-standardisation vs. unsafe-correction distinction: casing-only variants with an obvious approved mapping use `Standardise and Log`; anything ambiguous (suspicious spelling, unconfirmed thresholds, unresolved business meaning) uses `Pending Client Decision` rather than being guessed.

In [50]:
rows_checked_by_file = {
    "orders.csv": len(orders_df),
    "payments.csv": len(payments_df),
    "customers.csv": len(customers_df),
    "products.csv": len(products_df),
    "stores.csv": len(stores_df),
}
evidence_counts_by_rule_name = data_issues_log_df.groupby("proposed_rule").size()

candidate_rule_specs = [
    # orders.csv
    dict(file="orders.csv", column_or_columns="order_id", rule_name="order_id must be present",
         rule_description="order_id must not be blank or null.", quality_dimension="Completeness",
         severity="Critical", failure_action="Quarantine",
         business_impact="A missing order_id would make an order row unjoinable and uncountable in any downstream fact table.",
         client_confirmation_required="No", implementation_status="Confirmed"),
    dict(file="orders.csv", column_or_columns="order_id", rule_name="order_id must be unique",
         rule_description="order_id must not repeat across orders.csv rows.", quality_dimension="Uniqueness",
         severity="Critical", failure_action="Quarantine",
         business_impact="Duplicate order_id values can double-count revenue and break one-to-one joins with payments.",
         client_confirmation_required="No", implementation_status="Confirmed"),
    dict(file="orders.csv", column_or_columns="order_ts", rule_name="order_ts must be present",
         rule_description="order_ts must not be blank or null.", quality_dimension="Completeness",
         severity="Critical", failure_action="Quarantine",
         business_impact="A missing order timestamp prevents freshness checks and any time-based reporting for the order.",
         client_confirmation_required="No", implementation_status="Confirmed"),
    dict(file="orders.csv", column_or_columns="order_ts", rule_name="order_ts must parse as a valid timestamp",
         rule_description="order_ts must parse as a valid date-time value.", quality_dimension="Validity",
         severity="Critical", failure_action="Quarantine",
         business_impact="An unparseable timestamp cannot be ordered chronologically or used in any date-based KPI.",
         client_confirmation_required="No", implementation_status="Confirmed"),
    dict(file="orders.csv", column_or_columns="order_ts", rule_name="order_ts must not be future-dated",
         rule_description=f"order_ts must not be later than the fixed profiling reference date ({future_reference_date.date()}).",
         quality_dimension="Timeliness", severity="Critical", failure_action="Pending Client Decision",
         business_impact="Future-dated orders distort daily/weekly revenue trends and freshness metrics if treated as real transactions.",
         client_confirmation_required="Yes", implementation_status="Candidate"),
    dict(file="orders.csv", column_or_columns="customer_id", rule_name="customer_id must be present",
         rule_description="customer_id must not be blank or null.", quality_dimension="Completeness",
         severity="Pending Client Decision", failure_action="Pending Client Decision",
         business_impact="Whether guest/anonymous checkout is allowed determines if these orders can be used for customer-level reporting.",
         client_confirmation_required="Yes", implementation_status="Pending Client Decision"),
    dict(file="orders.csv", column_or_columns="product_id", rule_name="product_id must exist in products.csv",
         rule_description="product_id must match a product_id present in products.csv.", quality_dimension="Consistency",
         severity="Critical", failure_action="Quarantine",
         business_impact="An order referencing an unknown product cannot be enriched with category, brand, or price for reporting.",
         client_confirmation_required="No", implementation_status="Confirmed"),
    dict(file="orders.csv", column_or_columns="store_id", rule_name="store_id must exist in stores.csv",
         rule_description="store_id must match a store_id present in stores.csv.", quality_dimension="Consistency",
         severity="Critical", failure_action="Quarantine",
         business_impact="An order referencing an unknown store cannot be attributed to a zone/area for store-level reporting.",
         client_confirmation_required="No", implementation_status="Confirmed"),
    dict(file="orders.csv", column_or_columns="quantity", rule_name="quantity must be greater than zero",
         rule_description="quantity must be a positive number.", quality_dimension="Accuracy",
         severity="Pending Client Decision", failure_action="Pending Client Decision",
         business_impact="Negative amounts were also observed alongside non-positive quantities, which directly affects revenue and margin totals.",
         client_confirmation_required="Yes", implementation_status="Pending Client Decision"),
    dict(file="orders.csv", column_or_columns="discount_pct", rule_name="discount_pct must be between 0 and 100",
         rule_description="discount_pct must fall within the 0-100 range.", quality_dimension="Validity",
         severity="Critical", failure_action="Quarantine",
         business_impact="A discount outside 0-100 makes the order_amount formula produce a nonsensical result.",
         client_confirmation_required="No", implementation_status="Confirmed"),
    dict(file="orders.csv", column_or_columns="quantity,unit_price,discount_pct,order_amount",
         rule_name="order_amount must match the business formula",
         rule_description="order_amount must equal quantity * unit_price * (1 - discount_pct / 100), rounded to 2 decimals.",
         quality_dimension="Accuracy", severity="Critical", failure_action="Quarantine",
         business_impact="A formula mismatch means the recorded revenue for the order cannot be trusted without investigation.",
         client_confirmation_required="No", implementation_status="Confirmed"),
    dict(file="orders.csv", column_or_columns="order_status", rule_name="order_status must be in the approved list",
         rule_description=f"order_status must be one of {sorted(allowed_statuses)}.", quality_dimension="Validity",
         severity="Critical", failure_action="Quarantine",
         business_impact="An unapproved status value can be silently excluded from, or wrongly included in, status-based KPIs.",
         client_confirmation_required="No", implementation_status="Confirmed"),
    dict(file="orders.csv", column_or_columns="payment_mode", rule_name="payment_mode must be in the approved list",
         rule_description=f"payment_mode must be one of {sorted(allowed_payment_modes)}.", quality_dimension="Validity",
         severity="Warning", failure_action="Pending Client Decision",
         business_impact="Unapproved values split payment-mode reporting; some (e.g. trailing-space 'UPI ') are safe to trim, others (e.g. 'Cashh') are suspicious spellings with no approved mapping yet.",
         client_confirmation_required="Yes", implementation_status="Candidate"),
    dict(file="orders.csv", column_or_columns="delivery_minutes", rule_name="DELIVERED orders must have delivery_minutes",
         rule_description="Every order with order_status = DELIVERED must have a non-blank delivery_minutes value.",
         quality_dimension="Completeness", severity="Critical", failure_action="Quarantine",
         business_impact="Delivery-time KPIs (including the 10-minute promise) silently undercount these DELIVERED orders.",
         client_confirmation_required="No", implementation_status="Confirmed"),
    dict(file="orders.csv", column_or_columns="delivery_minutes", rule_name="delivery_minutes must not exceed the approved threshold",
         rule_description="delivery_minutes must not exceed an approved SLA threshold (observed values go as high as 1440 minutes; no threshold is approved yet).",
         quality_dimension="Accuracy", severity="Pending Client Decision", failure_action="Pending Client Decision",
         business_impact="Extreme delivery times distort average delivery time and the 10-minute promise KPI if left unflagged.",
         client_confirmation_required="Yes", implementation_status="Pending Client Decision"),
    # payments.csv
    dict(file="payments.csv", column_or_columns="payment_id", rule_name="payment_id must be unique",
         rule_description="payment_id must not repeat across payments.csv rows.", quality_dimension="Uniqueness",
         severity="Critical", failure_action="Quarantine",
         business_impact="Duplicate payment_id values can double-count settled revenue during reconciliation.",
         client_confirmation_required="No", implementation_status="Confirmed"),
    dict(file="payments.csv", column_or_columns="order_id", rule_name="payment order_id must exist in orders.csv",
         rule_description="payments.order_id must match an order_id present in orders.csv.", quality_dimension="Consistency",
         severity="Critical", failure_action="Quarantine",
         business_impact="A payment with no matching order cannot be reconciled against any order_amount and may represent lost or mismatched source data.",
         client_confirmation_required="No", implementation_status="Confirmed"),
    dict(file="payments.csv", column_or_columns="amount_paid", rule_name="SUCCESS payment amount must equal order_amount",
         rule_description="For payment_status = SUCCESS, amount_paid must equal the related order_amount (tolerance 0.01).",
         quality_dimension="Consistency", severity="Critical", failure_action="Quarantine",
         business_impact="A mismatch between a successful payment and its order amount directly affects finance reconciliation.",
         client_confirmation_required="No", implementation_status="Confirmed"),
    dict(file="payments.csv", column_or_columns="payment_ts", rule_name="payment_ts must not be future-dated",
         rule_description=f"payment_ts must not be later than the fixed profiling reference date ({future_reference_date.date()}).",
         quality_dimension="Timeliness", severity="Critical", failure_action="Pending Client Decision",
         business_impact="Future-dated payments distort daily settlement and reconciliation reporting if treated as real transactions.",
         client_confirmation_required="Yes", implementation_status="Candidate"),
    dict(file="payments.csv", column_or_columns="gateway_ref", rule_name="gateway_ref must be present for non-Cash-on-Delivery payments",
         rule_description="gateway_ref must not be blank when payment_mode is not Cash on Delivery.",
         quality_dimension="Completeness", severity="Warning", failure_action="Load and Log Warning",
         business_impact="A missing gateway reference on an electronic payment would prevent settlement tracing with the payment gateway; Cash on Delivery payments are expected to have none.",
         client_confirmation_required="No", implementation_status="Confirmed"),
    # customers.csv
    dict(file="customers.csv", column_or_columns="customer_id", rule_name="customer_id must be unique in customers.csv",
         rule_description="customer_id must not repeat across customers.csv rows.", quality_dimension="Uniqueness",
         severity="Critical", failure_action="Quarantine",
         business_impact="Duplicate customer_id values (with differing phone/area/pincode) make it unclear which contact details are current for that customer.",
         client_confirmation_required="No", implementation_status="Confirmed"),
    dict(file="customers.csv", column_or_columns="city", rule_name="city should not be blank",
         rule_description="city should be populated for every customer.", quality_dimension="Completeness",
         severity="Warning", failure_action="Load and Log Warning",
         business_impact="City-level reporting undercounts customers with a blank city, though the record may still load.",
         client_confirmation_required="No", implementation_status="Confirmed"),
    dict(file="customers.csv", column_or_columns="signup_date", rule_name="signup_date must not be future-dated",
         rule_description=f"signup_date must not be later than the fixed profiling reference date ({future_reference_date.date()}).",
         quality_dimension="Timeliness", severity="Warning", failure_action="Pending Client Decision",
         business_impact="A future signup_date is implausible and may indicate a source-system timestamp problem, but it does not affect revenue.",
         client_confirmation_required="Yes", implementation_status="Candidate"),
    dict(file="customers.csv", column_or_columns="loyalty_tier", rule_name="loyalty_tier must be in the approved list",
         rule_description="loyalty_tier must be one of {Bronze, Silver, Gold, Platinum}; casing variants (e.g. BRONZE, bronze, SILVER) should be standardised to the approved spelling.",
         quality_dimension="Validity", severity="Warning", failure_action="Standardise and Log",
         business_impact="Casing variants split loyalty-tier reporting into extra groups unless standardised to the approved spelling.",
         client_confirmation_required="No", implementation_status="Candidate"),
    # products.csv
    dict(file="products.csv", column_or_columns="product_id", rule_name="product_id must be unique in products.csv",
         rule_description="product_id must not repeat across products.csv rows.", quality_dimension="Uniqueness",
         severity="Critical", failure_action="Quarantine",
         business_impact="A reused product_id for a renamed/re-priced '(old)' variant makes it ambiguous which name/price applies to historical orders.",
         client_confirmation_required="No", implementation_status="Confirmed"),
    dict(file="products.csv", column_or_columns="category", rule_name="category should not be blank",
         rule_description="category should be populated for every product.", quality_dimension="Completeness",
         severity="Warning", failure_action="Load and Log Warning",
         business_impact="Category-level reporting undercounts products with a blank category, though the record may still load.",
         client_confirmation_required="No", implementation_status="Confirmed"),
    dict(file="products.csv", column_or_columns="mrp", rule_name="mrp must be greater than zero",
         rule_description="mrp must be a positive number.", quality_dimension="Accuracy",
         severity="Critical", failure_action="Pending Client Decision",
         business_impact="A zero or negative retail price is not usable in any margin or revenue calculation involving this product.",
         client_confirmation_required="Yes", implementation_status="Candidate"),
    dict(file="products.csv", column_or_columns="cost_price,mrp", rule_name="cost_price should not exceed mrp",
         rule_description="cost_price should not be greater than mrp.", quality_dimension="Accuracy",
         severity="Warning", failure_action="Pending Client Decision",
         business_impact="cost_price above mrp implies a negative gross margin on every sale, which needs catalog-team review.",
         client_confirmation_required="Yes", implementation_status="Candidate"),
    # stores.csv
    dict(file="stores.csv", column_or_columns="store_id", rule_name="store_id must be unique in stores.csv",
         rule_description="store_id must not repeat across stores.csv rows.", quality_dimension="Uniqueness",
         severity="Critical", failure_action="Quarantine",
         business_impact="No duplicate store_id values currently exist; this safeguard rule prevents a future load from silently introducing one.",
         client_confirmation_required="No", implementation_status="Confirmed"),
    dict(file="stores.csv", column_or_columns="zone", rule_name="zone should not be blank",
         rule_description="zone should be populated for every store.", quality_dimension="Completeness",
         severity="Warning", failure_action="Load and Log Warning",
         business_impact="Zone-level reporting undercounts stores with a blank zone, though the record may still load.",
         client_confirmation_required="No", implementation_status="Confirmed"),
    dict(file="stores.csv", column_or_columns="capacity_orders_per_day", rule_name="capacity_orders_per_day must be greater than zero",
         rule_description="capacity_orders_per_day must be a positive number.", quality_dimension="Accuracy",
         severity="Warning", failure_action="Pending Client Decision",
         business_impact="A zero or negative daily capacity cannot be used in any store-level load-balancing or operations calculation.",
         client_confirmation_required="Yes", implementation_status="Candidate"),
]

candidate_validation_rules_rows = []
for rule_index, rule_spec in enumerate(candidate_rule_specs, start=1):
    rows_checked = rows_checked_by_file[rule_spec["file"]]
    evidence_rows_failed = int(evidence_counts_by_rule_name.get(rule_spec["rule_name"], 0))
    candidate_validation_rules_rows.append(
        {
            "rule_id": f"RULE-{rule_index:03d}",
            "file": rule_spec["file"],
            "column_or_columns": rule_spec["column_or_columns"],
            "rule_name": rule_spec["rule_name"],
            "rule_description": rule_spec["rule_description"],
            "quality_dimension": rule_spec["quality_dimension"],
            "severity": rule_spec["severity"],
            "failure_action": rule_spec["failure_action"],
            "evidence_rows_failed": evidence_rows_failed,
            "evidence_summary": f"{evidence_rows_failed} of {rows_checked} {rule_spec['file']} rows fail this rule based on the Part 0 profiling evidence.",
            "business_impact": rule_spec["business_impact"],
            "client_confirmation_required": rule_spec["client_confirmation_required"],
            "implementation_status": rule_spec["implementation_status"],
        }
    )

candidate_validation_rules_df = pd.DataFrame(candidate_validation_rules_rows)

allowed_failure_actions = {"Quarantine", "Load and Log Warning", "Standardise and Log", "Pending Client Decision"}
allowed_implementation_statuses = {"Confirmed", "Candidate", "Pending Client Decision"}
assert candidate_validation_rules_df["rule_id"].is_unique
assert set(candidate_validation_rules_df["quality_dimension"]).issubset(allowed_quality_dimensions)
assert set(candidate_validation_rules_df["severity"]).issubset(allowed_severities)
assert set(candidate_validation_rules_df["failure_action"]).issubset(allowed_failure_actions)
assert set(candidate_validation_rules_df["implementation_status"]).issubset(allowed_implementation_statuses)

with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.width", None):
    print("Complete Candidate Validation Rules table:")
    display(candidate_validation_rules_df)

print("Rules by quality dimension:")
display(candidate_validation_rules_df.groupby("quality_dimension", sort=False).size().rename("rule_count").reset_index())

print("Rules by severity:")
display(candidate_validation_rules_df.groupby("severity", sort=False).size().rename("rule_count").reset_index())

rules_requiring_client_confirmation = candidate_validation_rules_df.loc[
    candidate_validation_rules_df["client_confirmation_required"] == "Yes"
]
print(f"Rules requiring client confirmation ({len(rules_requiring_client_confirmation)}):")
display(rules_requiring_client_confirmation[["rule_id", "file", "rule_name", "implementation_status"]])

candidate_validation_rules_df.to_csv(part0_outputs_dir / "part0_candidate_validation_rules.csv", index=False)
print(f"Exported: {(part0_outputs_dir / 'part0_candidate_validation_rules.csv').resolve()}")

Complete Candidate Validation Rules table:


,rule_id,file,column_or_columns,rule_name,rule_description,quality_dimension,severity,failure_action,evidence_rows_failed,evidence_summary,business_impact,client_confirmation_required,implementation_status
0,RULE-001,orders.csv,order_id,order_id must be present,order_id must not be blank or null.,Completeness,Critical,Quarantine,0,0 of 2412 orders.csv rows fail this rule based...,A missing order_id would make an order row unj...,No,Confirmed
1,RULE-002,orders.csv,order_id,order_id must be unique,order_id must not repeat across orders.csv rows.,Uniqueness,Critical,Quarantine,24,24 of 2412 orders.csv rows fail this rule base...,Duplicate order_id values can double-count rev...,No,Confirmed
2,RULE-003,orders.csv,order_ts,order_ts must be present,order_ts must not be blank or null.,Completeness,Critical,Quarantine,1,1 of 2412 orders.csv rows fail this rule based...,A missing order timestamp prevents freshness c...,No,Confirmed
3,RULE-004,orders.csv,order_ts,order_ts must parse as a valid timestamp,order_ts must parse as a valid date-time value.,Validity,Critical,Quarantine,5,5 of 2412 orders.csv rows fail this rule based...,An unparseable timestamp cannot be ordered chr...,No,Confirmed
4,RULE-005,orders.csv,order_ts,order_ts must not be future-dated,order_ts must not be later than the fixed prof...,Timeliness,Critical,Pending Client Decision,7,7 of 2412 orders.csv rows fail this rule based...,Future-dated orders distort daily/weekly reven...,Yes,Candidate
5,RULE-006,orders.csv,customer_id,customer_id must be present,customer_id must not be blank or null.,Completeness,Pending Client Decision,Pending Client Decision,18,18 of 2412 orders.csv rows fail this rule base...,Whether guest/anonymous checkout is allowed de...,Yes,Pending Client Decision
6,RULE-007,orders.csv,product_id,product_id must exist in products.csv,product_id must match a product_id present in ...,Consistency,Critical,Quarantine,9,9 of 2412 orders.csv rows fail this rule based...,An order referencing an unknown product cannot...,No,Confirmed
7,RULE-008,orders.csv,store_id,store_id must exist in stores.csv,store_id must match a store_id present in stor...,Consistency,Critical,Quarantine,5,5 of 2412 orders.csv rows fail this rule based...,An order referencing an unknown store cannot b...,No,Confirmed
8,RULE-009,orders.csv,quantity,quantity must be greater than zero,quantity must be a positive number.,Accuracy,Pending Client Decision,Pending Client Decision,10,10 of 2412 orders.csv rows fail this rule base...,Negative amounts were also observed alongside ...,Yes,Pending Client Decision
9,RULE-010,orders.csv,discount_pct,discount_pct must be between 0 and 100,discount_pct must fall within the 0-100 range.,Validity,Critical,Quarantine,3,3 of 2412 orders.csv rows fail this rule based...,A discount outside 0-100 makes the order_amoun...,No,Confirmed


Rules by quality dimension:


,quality_dimension,rule_count
0,Completeness,8
1,Uniqueness,5
2,Validity,5
3,Timeliness,3
4,Consistency,4
5,Accuracy,6


Rules by severity:


,severity,rule_count
0,Critical,19
1,Pending Client Decision,3
2,Warning,9


Rules requiring client confirmation (10):


,rule_id,file,rule_name,implementation_status
4,RULE-005,orders.csv,order_ts must not be future-dated,Candidate
5,RULE-006,orders.csv,customer_id must be present,Pending Client Decision
8,RULE-009,orders.csv,quantity must be greater than zero,Pending Client Decision
12,RULE-013,orders.csv,payment_mode must be in the approved list,Candidate
14,RULE-015,orders.csv,delivery_minutes must not exceed the approved ...,Pending Client Decision
18,RULE-019,payments.csv,payment_ts must not be future-dated,Candidate
22,RULE-023,customers.csv,signup_date must not be future-dated,Candidate
26,RULE-027,products.csv,mrp must be greater than zero,Candidate
27,RULE-028,products.csv,cost_price should not exceed mrp,Candidate
30,RULE-031,stores.csv,capacity_orders_per_day must be greater than zero,Candidate


Exported: /Users/irfan_shaik@optum.com/Downloads/IIT Roorkee/Assignment_2/IIT_Roorkie_CEC_Futurenece_Data_Pipeline_to_ML_Model/assignment/nammamart_dataset/part0_outputs/part0_candidate_validation_rules.csv


## Deliverable 3: Three Best Client Questions

`client_questions_df` holds exactly the three questions below. Each `affected_rules` value is looked up from `candidate_validation_rules_df["rule_id"]` by matching on `rule_name`, so the mapping always reflects the rules actually created above rather than being retyped by hand.

In [51]:
def rule_id_for(rule_name):
    matches = candidate_validation_rules_df.loc[
        candidate_validation_rules_df["rule_name"] == rule_name, "rule_id"
    ]
    assert len(matches) == 1, f"Expected exactly one rule_id for {rule_name!r}, found {len(matches)}."
    return matches.iloc[0]


client_questions_rows = [
    {
        "question_id": "Q-001",
        "client_question": (
            "Do zero or negative quantities represent returns, corrections, removed order lines, "
            "or source-system errors?"
        ),
        "evidence": (
            f"quantity <= 0 was observed in {int(quantity_failure_mask.sum())} of {len(orders_df)} orders.csv rows "
            f"(minimum quantity = {int(quantity_numeric.min())}); negative order_amount values were also observed "
            f"(minimum order_amount = {source_amount_numeric.min():.2f})."
        ),
        "why_it_matters": (
            "The answer affects revenue totals, gross margin, and whether these rows should be quarantined "
            "from finance reporting."
        ),
        "affected_rules": rule_id_for("quantity must be greater than zero"),
        "proposed_temporary_treatment": (
            "Exclude or quarantine these rows from finance reporting until the meaning is confirmed. "
            "Do not convert negative quantities to positive values."
        ),
        "suggested_business_owner": "Operations and Finance",
    },
    {
        "question_id": "Q-002",
        "client_question": "Are guest checkout or anonymous orders permitted (orders with no customer_id)?",
        "evidence": (
            f"customer_id is blank in {int(customer_id_blank_mask.sum())} of {len(orders_df)} orders.csv rows, "
            "spanning both the App and Web channels and multiple order statuses, so the rows cannot be "
            "explained by a single channel or outcome."
        ),
        "why_it_matters": (
            "These orders may still be usable for store-level revenue, but they cannot be used reliably for "
            "top-customer reporting. The answer determines whether a missing customer_id is Critical or Warning."
        ),
        "affected_rules": rule_id_for("customer_id must be present"),
        "proposed_temporary_treatment": (
            "Keep the raw record. Exclude the row from customer-level analytics. Quarantine from the curated "
            "order fact only if customer_id is confirmed as mandatory."
        ),
        "suggested_business_owner": "Product/Operations and Customer Data Owner",
    },
    {
        "question_id": "Q-003",
        "client_question": (
            "Should delivery_minutes greater than 120 invalidate the order row, or remain a valid order with "
            "an SLA warning?"
        ),
        "evidence": (
            f"delivery_minutes greater than 120 was found in {int(delivery_over_120_mask.sum())} of {len(orders_df)} "
            f"orders.csv rows; the maximum observed value was {int(delivery_numeric.max())} minutes."
        ),
        "why_it_matters": (
            "These values distort average delivery time and the 10-minute promise KPI. The answer determines "
            "Critical versus Warning severity for this rule."
        ),
        "affected_rules": rule_id_for("delivery_minutes must not exceed the approved threshold"),
        "proposed_temporary_treatment": (
            "Keep the order for revenue if all financial checks pass. Exclude or flag the duration in SLA "
            "reporting. Log the row as a warning until the threshold is approved."
        ),
        "suggested_business_owner": "Operations",
    },
]

client_questions_df = pd.DataFrame(client_questions_rows)[
    [
        "question_id", "client_question", "evidence", "why_it_matters",
        "affected_rules", "proposed_temporary_treatment", "suggested_business_owner",
    ]
]

assert len(client_questions_df) == 3, "Exactly three client questions are required."

# Every Pending Client Decision rule should map to one of these three questions.
pending_rules = candidate_validation_rules_df.loc[
    candidate_validation_rules_df["implementation_status"] == "Pending Client Decision", "rule_id"
]
unmapped_pending_rules = set(pending_rules) - set(client_questions_df["affected_rules"])
assert not unmapped_pending_rules, (
    f"Pending Client Decision rules not mapped to a client question: {unmapped_pending_rules}"
)

with pd.option_context("display.max_colwidth", None):
    print("Three best client questions:")
    display(client_questions_df)

client_questions_df.to_csv(part0_outputs_dir / "part0_client_questions.csv", index=False)
print(f"Exported: {(part0_outputs_dir / 'part0_client_questions.csv').resolve()}")

Three best client questions:


,question_id,client_question,evidence,why_it_matters,affected_rules,proposed_temporary_treatment,suggested_business_owner
0,Q-001,"Do zero or negative quantities represent returns, corrections, removed order lines, or source-system errors?",quantity <= 0 was observed in 10 of 2412 orders.csv rows (minimum quantity = -3); negative order_amount values were also observed (minimum order_amount = -1593.00).,"The answer affects revenue totals, gross margin, and whether these rows should be quarantined from finance reporting.",RULE-009,Exclude or quarantine these rows from finance reporting until the meaning is confirmed. Do not convert negative quantities to positive values.,Operations and Finance
1,Q-002,Are guest checkout or anonymous orders permitted (orders with no customer_id)?,"customer_id is blank in 18 of 2412 orders.csv rows, spanning both the App and Web channels and multiple order statuses, so the rows cannot be explained by a single channel or outcome.","These orders may still be usable for store-level revenue, but they cannot be used reliably for top-customer reporting. The answer determines whether a missing customer_id is Critical or Warning.",RULE-006,Keep the raw record. Exclude the row from customer-level analytics. Quarantine from the curated order fact only if customer_id is confirmed as mandatory.,Product/Operations and Customer Data Owner
2,Q-003,"Should delivery_minutes greater than 120 invalidate the order row, or remain a valid order with an SLA warning?",delivery_minutes greater than 120 was found in 6 of 2412 orders.csv rows; the maximum observed value was 1440 minutes.,These values distort average delivery time and the 10-minute promise KPI. The answer determines Critical versus Warning severity for this rule.,RULE-015,Keep the order for revenue if all financial checks pass. Exclude or flag the duration in SLA reporting. Log the row as a warning until the threshold is approved.,Operations


Exported: /Users/irfan_shaik@optum.com/Downloads/IIT Roorkee/Assignment_2/IIT_Roorkie_CEC_Futurenece_Data_Pipeline_to_ML_Model/assignment/nammamart_dataset/part0_outputs/part0_client_questions.csv


## Final Part 0 Summary

The summary below is generated from the same computed DataFrames used for the exports above (not retyped by hand), so every count is guaranteed to match the Data Issues Log, the Candidate Validation Rules table, and the referential-integrity/payment-reconciliation results.

In [52]:
from IPython.display import Markdown

total_source_rows_profiled = sum(len(dataframe) for dataframe in dataframes.values())

top_critical_findings = (
    data_issues_log_df.loc[data_issues_log_df["severity"] == "Critical"]
    .groupby(["file", "proposed_rule"], sort=False)
    .size()
    .rename("instances")
    .reset_index()
    .sort_values("instances", ascending=False)
    .head(5)
)
top_findings_lines = "\n".join(
    f"- **{row['file']} - {row['proposed_rule']}:** {row['instances']} rows."
    for _, row in top_critical_findings.iterrows()
)

referential_lines = "\n".join(
    f"- `{row['relationship']}`: {row['rows_failed']} of {row['non_blank_keys_checked']} non-blank keys failed "
    f"({row['blank_foreign_keys']} blank foreign keys), status **{row['status']}**."
    for _, row in referential_integrity_profile_df.iterrows()
)

rule_status_counts = candidate_validation_rules_df["implementation_status"].value_counts()

final_summary_markdown = f"""# Part 0: Discovery and Data Profiling - Final Summary

**1. Datasets profiled:** all six source files ({", ".join(file_names.values())}).

**2. Total source rows profiled:** {total_source_rows_profiled} rows across all six files.

**3. Most important confirmed findings (top Critical issues by row count):**
{top_findings_lines}

**4. Referential-integrity findings:**
{referential_lines}

**5. Payment-reconciliation findings:** {orphan_payment_count} payments reference an order_id that does not exist in
orders.csv (orphan payments, logged as Consistency issues), and {int(amount_mismatch_mask.sum())} additional
SUCCESS payments (matched to a real order) have an amount_paid that does not equal order_amount. Orphan payments
are never also counted as amount mismatches.

**6. Number of validation rules proposed:** {len(candidate_validation_rules_df)}
 ({rule_status_counts.get('Confirmed', 0)} Confirmed, {rule_status_counts.get('Candidate', 0)} Candidate,
 {rule_status_counts.get('Pending Client Decision', 0)} Pending Client Decision).

**7. The three client questions:**
{chr(10).join(f"- **{row['question_id']}:** {row['client_question']}" for _, row in client_questions_df.iterrows())}

**8. Readiness statement:** the {rule_status_counts.get('Confirmed', 0)} rules marked `Confirmed` above
(structural keys, foreign keys, the order-amount formula, allowed-value lists for order_status/discount, and
payment reconciliation) are ready for ETL/ELT implementation as quarantine or warning checks. The
{rule_status_counts.get('Candidate', 0)} `Candidate` rules (future-dated timestamps, casing standardisation,
product pricing anomalies, capacity checks) are reasonable to implement but have not been explicitly confirmed
by the client. The {rule_status_counts.get('Pending Client Decision', 0)} `Pending Client Decision` rules
(quantity <= 0, missing customer_id, delivery_minutes > 120) require the client answers captured in
`client_questions_df` before their severity and treatment can be finalised.

## Coverage checklist

- [x] All six files profiled
- [x] Row counts completed
- [x] Missing values completed
- [x] Duplicate keys completed
- [x] Numeric min/max completed
- [x] Date min/max completed
- [x] Category profiling completed
- [x] Four referential-integrity checks completed
- [x] Client business rules checked
- [x] Data Issues Log created ({len(data_issues_log_df)} rows, {total_distinct_affected_rows} distinct affected source rows)
- [x] Candidate Validation Rules created ({len(candidate_validation_rules_df)} rules)
- [x] Three client questions created
- [x] Output CSV files exported
"""

display(Markdown(final_summary_markdown))

# Part 0: Discovery and Data Profiling - Final Summary

**1. Datasets profiled:** all six source files (customers.csv, orders.csv, payments.csv, products.csv, stores.csv, orders_2026-10-01_increment.csv).

**2. Total source rows profiled:** 5372 rows across all six files.

**3. Most important confirmed findings (top Critical issues by row count):**
- **orders.csv - order_id must be unique:** 24 rows.
- **orders.csv - order_amount must match the business formula:** 18 rows.
- **payments.csv - SUCCESS payment amount must equal order_amount:** 14 rows.
- **customers.csv - customer_id must be unique in customers.csv:** 10 rows.
- **orders.csv - product_id must exist in products.csv:** 9 rows.

**4. Referential-integrity findings:**
- `orders.customer_id -> customers.customer_id`: 0 of 2394 non-blank keys failed (18 blank foreign keys), status **PASS**.
- `orders.product_id -> products.product_id`: 9 of 2412 non-blank keys failed (0 blank foreign keys), status **INVESTIGATE**.
- `orders.store_id -> stores.store_id`: 5 of 2412 non-blank keys failed (0 blank foreign keys), status **INVESTIGATE**.
- `payments.order_id -> orders.order_id`: 8 of 2228 non-blank keys failed (0 blank foreign keys), status **INVESTIGATE**.

**5. Payment-reconciliation findings:** 8 payments reference an order_id that does not exist in
orders.csv (orphan payments, logged as Consistency issues), and 14 additional
SUCCESS payments (matched to a real order) have an amount_paid that does not equal order_amount. Orphan payments
are never also counted as amount mismatches.

**6. Number of validation rules proposed:** 31
 (20 Confirmed, 8 Candidate,
 3 Pending Client Decision).

**7. The three client questions:**
- **Q-001:** Do zero or negative quantities represent returns, corrections, removed order lines, or source-system errors?
- **Q-002:** Are guest checkout or anonymous orders permitted (orders with no customer_id)?
- **Q-003:** Should delivery_minutes greater than 120 invalidate the order row, or remain a valid order with an SLA warning?

**8. Readiness statement:** the 20 rules marked `Confirmed` above
(structural keys, foreign keys, the order-amount formula, allowed-value lists for order_status/discount, and
payment reconciliation) are ready for ETL/ELT implementation as quarantine or warning checks. The
8 `Candidate` rules (future-dated timestamps, casing standardisation,
product pricing anomalies, capacity checks) are reasonable to implement but have not been explicitly confirmed
by the client. The 3 `Pending Client Decision` rules
(quantity <= 0, missing customer_id, delivery_minutes > 120) require the client answers captured in
`client_questions_df` before their severity and treatment can be finalised.

## Coverage checklist

- [x] All six files profiled
- [x] Row counts completed
- [x] Missing values completed
- [x] Duplicate keys completed
- [x] Numeric min/max completed
- [x] Date min/max completed
- [x] Category profiling completed
- [x] Four referential-integrity checks completed
- [x] Client business rules checked
- [x] Data Issues Log created (208 rows, 181 distinct affected source rows)
- [x] Candidate Validation Rules created (31 rules)
- [x] Three client questions created
- [x] Output CSV files exported


In [53]:
grouped_issue_summary = (
    data_issues_log_df.groupby(["file", "proposed_rule", "quality_dimension", "severity"], sort=False)
    .agg(
        affected_rows=("row_key", "nunique"),
        representative_row_keys=(
            "row_key",
            lambda keys: ", ".join(sorted(set(keys))[:3]) + (" ..." if len(set(keys)) > 3 else ""),
        ),
    )
    .reset_index()
    .sort_values(["file", "severity", "affected_rows"], ascending=[True, True, False])
)

issue_summary_table_lines = "\n".join(
    f"| {row['file']} | {row['proposed_rule']} | {row['quality_dimension']} | {row['severity']} | "
    f"{row['affected_rows']} | {row['representative_row_keys']} |"
    for _, row in grouped_issue_summary.iterrows()
)

rules_table_lines = "\n".join(
    f"| {row['rule_id']} | {row['file']} | {row['rule_name']} | {row['severity']} | "
    f"{row['implementation_status']} | {row['evidence_rows_failed']} |"
    for _, row in candidate_validation_rules_df.iterrows()
)

questions_lines = "\n\n".join(
    f"**{row['question_id']}. {row['client_question']}**\n\n"
    f"- Evidence: {row['evidence']}\n"
    f"- Why it matters: {row['why_it_matters']}\n"
    f"- Proposed temporary treatment: {row['proposed_temporary_treatment']}\n"
    f"- Suggested business owner: {row['suggested_business_owner']}"
    for _, row in client_questions_df.iterrows()
)

submission_section_markdown = f"""## Part 0: Discovery and Data Profiling

### Profiling Summary

All six source files ({", ".join(file_names.values())}) were profiled across six stages: initial inspection,
structural profiling (missing values, duplicates, candidate keys, schema comparison), column profiling (numeric,
date, categorical), cross-column business rules, payment reconciliation, and referential integrity. A total of
{total_source_rows_profiled} source rows were profiled. No source row was cleaned, standardised, deduplicated,
quarantined, or deleted; every DataFrame used in this section remains exactly as loaded from the CSV files.

### Data Issues Log

`data_issues_log_df` contains {len(data_issues_log_df)} row-level issue instances covering
{total_distinct_affected_rows} distinct affected source rows across {data_issues_log_df["file"].nunique()} files.
The table below groups those instances by file, rule, quality dimension, and severity; representative affected
row keys are shown for each group (not the full list). The complete row-level log is available in
`part0_data_issues_log.csv`.

| file | rule | quality_dimension | severity | affected_rows | representative_row_keys |
|---|---|---|---|---:|---|
{issue_summary_table_lines}

### Candidate Validation Rules

{len(candidate_validation_rules_df)} candidate validation rules were derived from the confirmed profiling
findings above (one row per distinct validation concept, not per failing record). The full table, including
`rule_description`, `failure_action`, `business_impact`, and `client_confirmation_required`, is available in
`part0_candidate_validation_rules.csv`.

| rule_id | file | rule_name | severity | implementation_status | evidence_rows_failed |
|---|---|---|---|---|---:|
{rules_table_lines}

### Client Questions

{questions_lines}

### Part 0 Conclusion

The {int((candidate_validation_rules_df["implementation_status"] == "Confirmed").sum())} `Confirmed` rules above
are ready for ETL/ELT implementation as quarantine or warning checks. The
{int((candidate_validation_rules_df["implementation_status"] == "Candidate").sum())} `Candidate` rules are
reasonable to implement but have not been explicitly confirmed by the client. The
{int((candidate_validation_rules_df["implementation_status"] == "Pending Client Decision").sum())}
`Pending Client Decision` rules require the client's answers to the three questions above before their severity
and treatment can be finalised. Full detail for every deliverable is available in `part0_data_issues_log.csv`,
`part0_candidate_validation_rules.csv`, and `part0_client_questions.csv`.
"""

submission_section_path = part0_outputs_dir / "part0_submission_section.md"
submission_section_path.write_text(submission_section_markdown)
print(f"Exported: {submission_section_path.resolve()}")

Exported: /Users/irfan_shaik@optum.com/Downloads/IIT Roorkee/Assignment_2/IIT_Roorkie_CEC_Futurenece_Data_Pipeline_to_ML_Model/assignment/nammamart_dataset/part0_outputs/part0_submission_section.md


In [54]:
# --- Final Assertions ---------------------------------------------------

expected_output_files = [
    "part0_data_issues_log.csv",
    "part0_candidate_validation_rules.csv",
    "part0_client_questions.csv",
    "part0_submission_section.md",
]
for file_name in expected_output_files:
    output_path = part0_outputs_dir / file_name
    assert output_path.exists(), f"Expected output file missing: {output_path}"
    assert output_path.stat().st_size > 0, f"Expected output file is empty: {output_path}"
print("All expected output files exist and are non-empty.")

# No source DataFrame was modified: row counts must still match the Stage 1 dataset inventory.
original_row_counts = inventory_df.set_index("dataframe_name")["rows"].to_dict()
for dataframe_name, dataframe in dataframes.items():
    assert len(dataframe) == original_row_counts[dataframe_name], (
        f"{dataframe_name} row count changed from {original_row_counts[dataframe_name]} to {len(dataframe)}."
    )
print("No source DataFrame row count changed; no source DataFrame was modified in this stage.")

# The issue log contains no invented row keys (re-checked explicitly here, in addition to the Deliverable 1 assertions).
for file_name, (source_dataframe, key_column) in file_to_source.items():
    known_keys = set(source_dataframe[key_column].astype("string"))
    file_rows = data_issues_log_df.loc[data_issues_log_df["file"] == file_name, "row_key"]
    assert set(file_rows).issubset(known_keys), f"{file_name} contains a row_key not present in the source file."
print("Every row_key in data_issues_log_df exists in its stated source DataFrame.")

# Orphan payments are not double-counted as amount mismatches.
orphan_payment_ids = set(orphan_payment_rows_df["payment_id"])
amount_mismatch_payment_ids = set(
    data_issues_log_df.loc[
        data_issues_log_df["evidence_check"] == "Stage 4.1 payment reconciliation (SUCCESS amount mismatch)",
        "row_key",
    ]
)
assert orphan_payment_ids.isdisjoint(amount_mismatch_payment_ids), (
    "Orphan payments must never also be logged as SUCCESS amount mismatches."
)
print("Orphan payments and SUCCESS amount-mismatch payments are disjoint (no double counting).")

# Failure counts in the summaries match the detailed issue log.
assert int(issues_by_file["issue_instances"].sum()) == len(data_issues_log_df)
assert int(issues_by_quality_dimension["issue_instances"].sum()) == len(data_issues_log_df)
assert int(issues_by_severity["issue_instances"].sum()) == len(data_issues_log_df)
assert int(issues_by_evidence_check["issue_instances"].sum()) == len(data_issues_log_df)
print("Summary counts match the detailed Data Issues Log.")

# All quality dimensions and severities use only the approved names (re-checked here for both deliverables).
assert set(data_issues_log_df["quality_dimension"]).issubset(allowed_quality_dimensions)
assert set(candidate_validation_rules_df["quality_dimension"]).issubset(allowed_quality_dimensions)
assert set(data_issues_log_df["severity"]).issubset(allowed_severities)
assert set(candidate_validation_rules_df["severity"]).issubset(allowed_severities)
print("All quality_dimension and severity values use only the approved names.")

# Every proposed Critical rule has a clear reason for quarantine (a populated business_impact statement).
critical_rules = candidate_validation_rules_df.loc[candidate_validation_rules_df["severity"] == "Critical"]
assert critical_rules["business_impact"].astype("string").str.strip().ne("").all(), (
    "Every Critical rule must state a clear business-impact reason for quarantine."
)
print("Every Critical rule has a documented business-impact reason for quarantine.")

# Every Pending Client Decision rule maps to one of the three client questions (re-checked here for both deliverables).
pending_rules_check = candidate_validation_rules_df.loc[
    candidate_validation_rules_df["implementation_status"] == "Pending Client Decision", "rule_id"
]
assert set(pending_rules_check).issubset(set(client_questions_df["affected_rules"])), (
    "Every Pending Client Decision rule must map to one of the three client questions."
)
print("Every Pending Client Decision rule maps to one of the three client questions.")

print()
print("All Final Part 0 assertions passed. Stopping here, as instructed: Part A, ETL, Airflow, DuckDB, and ELT")
print("implementation are explicitly out of scope for this notebook.")

All expected output files exist and are non-empty.
No source DataFrame row count changed; no source DataFrame was modified in this stage.
Every row_key in data_issues_log_df exists in its stated source DataFrame.
Orphan payments and SUCCESS amount-mismatch payments are disjoint (no double counting).
Summary counts match the detailed Data Issues Log.
All quality_dimension and severity values use only the approved names.
Every Critical rule has a documented business-impact reason for quarantine.
Every Pending Client Decision rule maps to one of the three client questions.

All Final Part 0 assertions passed. Stopping here, as instructed: Part A, ETL, Airflow, DuckDB, and ELT
implementation are explicitly out of scope for this notebook.


In [55]:
from IPython.display import Markdown, display

submission_path = dataset_dir / "part0_outputs" / "part0_submission_section.md"
display(Markdown(submission_path.read_text()))
print(f"Markdown file generated at: {submission_path.resolve()}")

## Part 0: Discovery and Data Profiling

### Profiling Summary

All six source files (customers.csv, orders.csv, payments.csv, products.csv, stores.csv, orders_2026-10-01_increment.csv) were profiled across six stages: initial inspection,
structural profiling (missing values, duplicates, candidate keys, schema comparison), column profiling (numeric,
date, categorical), cross-column business rules, payment reconciliation, and referential integrity. A total of
5372 source rows were profiled. No source row was cleaned, standardised, deduplicated,
quarantined, or deleted; every DataFrame used in this section remains exactly as loaded from the CSV files.

### Data Issues Log

`data_issues_log_df` contains 208 row-level issue instances covering
181 distinct affected source rows across 5 files.
The table below groups those instances by file, rule, quality dimension, and severity; representative affected
row keys are shown for each group (not the full list). The complete row-level log is available in
`part0_data_issues_log.csv`.

| file | rule | quality_dimension | severity | affected_rows | representative_row_keys |
|---|---|---|---|---:|---|
| customers.csv | customer_id must be unique in customers.csv | Uniqueness | Critical | 5 | C00036, C00103, C00226 ... |
| customers.csv | loyalty_tier must be in the approved list | Validity | Warning | 8 | C00031, C00032, C00092 ... |
| customers.csv | city should not be blank | Completeness | Warning | 6 | C00142, C00219, C00233 ... |
| customers.csv | signup_date must not be future-dated | Timeliness | Warning | 3 | C00038, C00398, C00400 |
| orders.csv | order_amount must match the business formula | Accuracy | Critical | 18 | ORD000677, ORD000801, ORD000845 ... |
| orders.csv | order_id must be unique | Uniqueness | Critical | 12 | ORD000539, ORD000783, ORD001110 ... |
| orders.csv | product_id must exist in products.csv | Consistency | Critical | 9 | ORD000070, ORD000134, ORD000596 ... |
| orders.csv | DELIVERED orders must have delivery_minutes | Completeness | Critical | 8 | ORD000326, ORD000731, ORD000878 ... |
| orders.csv | order_ts must not be future-dated | Timeliness | Critical | 7 | ORD001089, ORD001121, ORD001693 ... |
| orders.csv | order_status must be in the approved list | Validity | Critical | 5 | ORD000153, ORD000500, ORD001066 ... |
| orders.csv | order_ts must parse as a valid timestamp | Validity | Critical | 5 | ORD000403, ORD000534, ORD000946 ... |
| orders.csv | store_id must exist in stores.csv | Consistency | Critical | 5 | ORD001046, ORD001275, ORD001492 ... |
| orders.csv | discount_pct must be between 0 and 100 | Validity | Critical | 3 | ORD001237, ORD001468, ORD002103 |
| orders.csv | order_ts must be present | Completeness | Critical | 1 | ORD000362 |
| orders.csv | customer_id must be present | Completeness | Pending Client Decision | 18 | ORD000022, ORD000580, ORD000583 ... |
| orders.csv | quantity must be greater than zero | Accuracy | Pending Client Decision | 10 | ORD000099, ORD000462, ORD000469 ... |
| orders.csv | payment_mode must be in the approved list | Validity | Warning | 10 | ORD000159, ORD000433, ORD000691 ... |
| orders.csv | delivery_minutes must not exceed the approved threshold | Accuracy | Warning | 6 | ORD000020, ORD000424, ORD000455 ... |
| payments.csv | SUCCESS payment amount must equal order_amount | Consistency | Critical | 14 | PAY0000044, PAY0000161, PAY0000323 ... |
| payments.csv | payment order_id must exist in orders.csv | Consistency | Critical | 8 | PAY0002217, PAY0002218, PAY0002219 ... |
| payments.csv | payment_ts must not be future-dated | Timeliness | Critical | 7 | PAY0000127, PAY0000267, PAY0000337 ... |
| payments.csv | payment_id must be unique | Uniqueness | Critical | 4 | PAY0000509, PAY0000755, PAY0001651 ... |
| products.csv | product_id must be unique in products.csv | Uniqueness | Critical | 2 | P0030, P0140 |
| products.csv | mrp must be greater than zero | Accuracy | Critical | 1 | P0109 |
| products.csv | cost_price should not exceed mrp | Accuracy | Warning | 5 | P0019, P0073, P0074 ... |
| products.csv | category should not be blank | Completeness | Warning | 3 | P0051, P0060, P0068 |
| stores.csv | zone should not be blank | Completeness | Warning | 1 | S07 |
| stores.csv | capacity_orders_per_day must be greater than zero | Accuracy | Warning | 1 | S12 |

### Candidate Validation Rules

31 candidate validation rules were derived from the confirmed profiling
findings above (one row per distinct validation concept, not per failing record). The full table, including
`rule_description`, `failure_action`, `business_impact`, and `client_confirmation_required`, is available in
`part0_candidate_validation_rules.csv`.

| rule_id | file | rule_name | severity | implementation_status | evidence_rows_failed |
|---|---|---|---|---|---:|
| RULE-001 | orders.csv | order_id must be present | Critical | Confirmed | 0 |
| RULE-002 | orders.csv | order_id must be unique | Critical | Confirmed | 24 |
| RULE-003 | orders.csv | order_ts must be present | Critical | Confirmed | 1 |
| RULE-004 | orders.csv | order_ts must parse as a valid timestamp | Critical | Confirmed | 5 |
| RULE-005 | orders.csv | order_ts must not be future-dated | Critical | Candidate | 7 |
| RULE-006 | orders.csv | customer_id must be present | Pending Client Decision | Pending Client Decision | 18 |
| RULE-007 | orders.csv | product_id must exist in products.csv | Critical | Confirmed | 9 |
| RULE-008 | orders.csv | store_id must exist in stores.csv | Critical | Confirmed | 5 |
| RULE-009 | orders.csv | quantity must be greater than zero | Pending Client Decision | Pending Client Decision | 10 |
| RULE-010 | orders.csv | discount_pct must be between 0 and 100 | Critical | Confirmed | 3 |
| RULE-011 | orders.csv | order_amount must match the business formula | Critical | Confirmed | 18 |
| RULE-012 | orders.csv | order_status must be in the approved list | Critical | Confirmed | 5 |
| RULE-013 | orders.csv | payment_mode must be in the approved list | Warning | Candidate | 10 |
| RULE-014 | orders.csv | DELIVERED orders must have delivery_minutes | Critical | Confirmed | 8 |
| RULE-015 | orders.csv | delivery_minutes must not exceed the approved threshold | Pending Client Decision | Pending Client Decision | 6 |
| RULE-016 | payments.csv | payment_id must be unique | Critical | Confirmed | 8 |
| RULE-017 | payments.csv | payment order_id must exist in orders.csv | Critical | Confirmed | 8 |
| RULE-018 | payments.csv | SUCCESS payment amount must equal order_amount | Critical | Confirmed | 14 |
| RULE-019 | payments.csv | payment_ts must not be future-dated | Critical | Candidate | 7 |
| RULE-020 | payments.csv | gateway_ref must be present for non-Cash-on-Delivery payments | Warning | Confirmed | 0 |
| RULE-021 | customers.csv | customer_id must be unique in customers.csv | Critical | Confirmed | 10 |
| RULE-022 | customers.csv | city should not be blank | Warning | Confirmed | 6 |
| RULE-023 | customers.csv | signup_date must not be future-dated | Warning | Candidate | 3 |
| RULE-024 | customers.csv | loyalty_tier must be in the approved list | Warning | Candidate | 8 |
| RULE-025 | products.csv | product_id must be unique in products.csv | Critical | Confirmed | 4 |
| RULE-026 | products.csv | category should not be blank | Warning | Confirmed | 3 |
| RULE-027 | products.csv | mrp must be greater than zero | Critical | Candidate | 1 |
| RULE-028 | products.csv | cost_price should not exceed mrp | Warning | Candidate | 5 |
| RULE-029 | stores.csv | store_id must be unique in stores.csv | Critical | Confirmed | 0 |
| RULE-030 | stores.csv | zone should not be blank | Warning | Confirmed | 1 |
| RULE-031 | stores.csv | capacity_orders_per_day must be greater than zero | Warning | Candidate | 1 |

### Client Questions

**Q-001. Do zero or negative quantities represent returns, corrections, removed order lines, or source-system errors?**

- Evidence: quantity <= 0 was observed in 10 of 2412 orders.csv rows (minimum quantity = -3); negative order_amount values were also observed (minimum order_amount = -1593.00).
- Why it matters: The answer affects revenue totals, gross margin, and whether these rows should be quarantined from finance reporting.
- Proposed temporary treatment: Exclude or quarantine these rows from finance reporting until the meaning is confirmed. Do not convert negative quantities to positive values.
- Suggested business owner: Operations and Finance

**Q-002. Are guest checkout or anonymous orders permitted (orders with no customer_id)?**

- Evidence: customer_id is blank in 18 of 2412 orders.csv rows, spanning both the App and Web channels and multiple order statuses, so the rows cannot be explained by a single channel or outcome.
- Why it matters: These orders may still be usable for store-level revenue, but they cannot be used reliably for top-customer reporting. The answer determines whether a missing customer_id is Critical or Warning.
- Proposed temporary treatment: Keep the raw record. Exclude the row from customer-level analytics. Quarantine from the curated order fact only if customer_id is confirmed as mandatory.
- Suggested business owner: Product/Operations and Customer Data Owner

**Q-003. Should delivery_minutes greater than 120 invalidate the order row, or remain a valid order with an SLA warning?**

- Evidence: delivery_minutes greater than 120 was found in 6 of 2412 orders.csv rows; the maximum observed value was 1440 minutes.
- Why it matters: These values distort average delivery time and the 10-minute promise KPI. The answer determines Critical versus Warning severity for this rule.
- Proposed temporary treatment: Keep the order for revenue if all financial checks pass. Exclude or flag the duration in SLA reporting. Log the row as a warning until the threshold is approved.
- Suggested business owner: Operations

### Part 0 Conclusion

The 20 `Confirmed` rules above
are ready for ETL/ELT implementation as quarantine or warning checks. The
8 `Candidate` rules are
reasonable to implement but have not been explicitly confirmed by the client. The
3
`Pending Client Decision` rules require the client's answers to the three questions above before their severity
and treatment can be finalised. Full detail for every deliverable is available in `part0_data_issues_log.csv`,
`part0_candidate_validation_rules.csv`, and `part0_client_questions.csv`.


Markdown file generated at: /Users/irfan_shaik@optum.com/Downloads/IIT Roorkee/Assignment_2/IIT_Roorkie_CEC_Futurenece_Data_Pipeline_to_ML_Model/assignment/nammamart_dataset/part0_outputs/part0_submission_section.md
